# Fase 3 - SRNet como 2º adversário
Adiciona a SRNet ao laço adversarial sobre o gerador residual da Fase 2.5.

## 1. GPU

In [ ]:
import torch
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print('SEM GPU! Vá em Ambiente de execução → Alterar o tipo de ambiente → GPU e reinicie.')


## 2. Google Drive

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/IC_stego'
except Exception as e:
    print('Drive não montado (', e, ') -> usando /content (TEMPORÁRIO, você perde tudo ao fim da sessão)')
    BASE = '/content/IC_stego'

PASTA_DADOS_BASE = os.path.join(BASE, 'datasets')
PASTA_SAIDA_BASE = os.path.join(BASE, 'runs')
os.makedirs(PASTA_DADOS_BASE, exist_ok=True); os.makedirs(PASTA_SAIDA_BASE, exist_ok=True)
PASTA_TRABALHO = '/content/runs'
os.makedirs(PASTA_TRABALHO, exist_ok=True)
print('Dataset ->', PASTA_DADOS_BASE)
print('Saída   ->', PASTA_SAIDA_BASE)


## 3. Código
Grava os módulos das Fases 0, 2, 2.5 e 3.

In [ ]:
%%writefile fase0_codificador_decodificador.py
import os
import glob
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image


CONFIG = {
  
    "pasta_dados": None,
    "canais": 3,
    "tamanho_img": 128,        
    "fracao_val": 0.05,       
    "max_imagens": None,      
    

    "profundidade_dados": 1,  
    "tamanho_msg": None,      


    "tamanho_lote": 16,       
    "epocas": 30,
    "taxa_aprend": 1e-3,      
    "largura": 32,             
    "lambda_imagem": 1.0,      
    "lambda_mensagem": 1.0,    
    "lambda_residuo": 0.0,    
    "usar_amp": True,          
    "alvo_bitacc": 0.99,       

    "num_trabalhadores": 2,   
    "semente": 42,             
    "pasta_saida": "runs/phase0",
    "salvar_cada": 1,        
    "dispositivo": "cuda" if torch.cuda.is_available() else "cpu",
}



COLAB_GPU = dict(
    tamanho_img=128,
    tamanho_msg=32,         
    tamanho_lote=32,
    epocas=120,
    largura=64,
    max_imagens=None,
    num_trabalhadores=4,
    lambda_imagem=2.0,
    lambda_residuo=0.05,
)


EXTENSOES_IMG = (".png", ".jpg", ".jpeg", ".bmp", ".pgm", ".ppm", ".tif", ".tiff", ".webp")


class DatasetImagens(Dataset):
    def __init__(self, caminhos, canais, tamanho_img, treino=True):
        self.caminhos = caminhos
        self.canais = canais
        self.modo = "L" if canais == 1 else "RGB"
        if treino:
            self.transformacao = transforms.Compose([
                transforms.Resize(tamanho_img),
                transforms.RandomCrop(tamanho_img, pad_if_needed=True),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
            ])
        else:
            self.transformacao = transforms.Compose([
                transforms.Resize(tamanho_img),
                transforms.CenterCrop(tamanho_img),
                transforms.ToTensor(),
            ])

    def __len__(self):
        return len(self.caminhos)

    def __getitem__(self, indice):
        imagem = Image.open(self.caminhos[indice]).convert(self.modo)
        return self.transformacao(imagem)


class DatasetSintetico(Dataset):
    def __init__(self, canais, tamanho_img, n=512):
        self.canais, self.tamanho_img, self.n = canais, tamanho_img, n

    def __len__(self):
        return self.n

    def __getitem__(self, indice):
        base = torch.rand(self.canais, 1, 1)
        gradiente = torch.linspace(0, 1, self.tamanho_img).view(1, 1, -1)
        return (base + 0.3 * gradiente).clamp(0, 1).expand(
            self.canais, self.tamanho_img, self.tamanho_img).clone()


def montar_loaders(cfg):
    if cfg["pasta_dados"]:
        caminhos = []
        for ext in EXTENSOES_IMG:
            caminhos += glob.glob(os.path.join(cfg["pasta_dados"], "**", "*" + ext), recursive=True)
            caminhos += glob.glob(os.path.join(cfg["pasta_dados"], "**", "*" + ext.upper()), recursive=True)
        caminhos = sorted(set(caminhos))
        if not caminhos:
            raise FileNotFoundError(f"Nenhuma imagem encontrada em {cfg['pasta_dados']}")
        random.Random(cfg["semente"]).shuffle(caminhos)
        if cfg.get("max_imagens"):
            caminhos = caminhos[:cfg["max_imagens"]]
            print(f"[dados] max_imagens={cfg['max_imagens']} -> usando {len(caminhos)} imagens")
        n_val = max(1, int(len(caminhos) * cfg["fracao_val"]))
        caminhos_val, caminhos_treino = caminhos[:n_val], caminhos[n_val:]
        print(f"[dados] {len(caminhos_treino)} treino / {len(caminhos_val)} val  ({cfg['pasta_dados']})")
        ds_treino = DatasetImagens(caminhos_treino, cfg["canais"], cfg["tamanho_img"], treino=True)
        ds_val = DatasetImagens(caminhos_val, cfg["canais"], cfg["tamanho_img"], treino=False)
    else:
        print("[dados] pasta_dados=None -> usando dataset SINTÉTICO (teste rápido)")
        ds_treino = DatasetSintetico(cfg["canais"], cfg["tamanho_img"], n=512)
        ds_val = DatasetSintetico(cfg["canais"], cfg["tamanho_img"], n=64)

    loader_treino = DataLoader(ds_treino, batch_size=cfg["tamanho_lote"], shuffle=True,
                               num_workers=cfg["num_trabalhadores"], drop_last=True, pin_memory=True)
    loader_val = DataLoader(ds_val, batch_size=cfg["tamanho_lote"], shuffle=False,
                            num_workers=cfg["num_trabalhadores"], pin_memory=True)
    return loader_treino, loader_val


def bloco_conv(canais_entrada, canais_saida):
    return nn.Sequential(
        nn.Conv2d(canais_entrada, canais_saida, 3, padding=1),
        nn.BatchNorm2d(canais_saida),
        nn.LeakyReLU(0.2, inplace=True),
    )


class Codificador(nn.Module):

    def __init__(self, canais, profundidade_dados, largura=32):
        super().__init__()
        self.bloco1 = bloco_conv(canais, largura)
        self.bloco2 = bloco_conv(largura + profundidade_dados, largura)
        self.bloco3 = bloco_conv(2 * largura + profundidade_dados, largura)
        self.para_residuo = nn.Conv2d(3 * largura + profundidade_dados, canais, 3, padding=1)

    def forward(self, portadora, mensagem):
        a = self.bloco1(portadora)
        b = self.bloco2(torch.cat([a, mensagem], dim=1))
        c = self.bloco3(torch.cat([a, b, mensagem], dim=1))
        residuo = self.para_residuo(torch.cat([a, b, c, mensagem], dim=1))
        estego = (portadora + residuo).clamp(0.0, 1.0)
        return estego


class Decodificador(nn.Module):
    

    def __init__(self, canais, profundidade_dados, largura=32):
        super().__init__()
        self.bloco1 = bloco_conv(canais, largura)
        self.bloco2 = bloco_conv(largura, largura)
        self.bloco3 = bloco_conv(2 * largura, largura)
        self.para_logits = nn.Conv2d(3 * largura, profundidade_dados, 3, padding=1)

    def forward(self, estego):
        a = self.bloco1(estego)
        b = self.bloco2(a)
        c = self.bloco3(torch.cat([a, b], dim=1))
        return self.para_logits(torch.cat([a, b, c], dim=1))


def _tamanho_msg(cfg):
    return cfg.get("tamanho_msg") or cfg["tamanho_img"]


def mensagem_aleatoria(tamanho_lote, cfg):
    tm = _tamanho_msg(cfg)
    return torch.randint(0, 2, (tamanho_lote, cfg["profundidade_dados"], tm, tm),
                         dtype=torch.float32, device=cfg["dispositivo"])


def espalhar_mensagem(mensagem, cfg):
    if _tamanho_msg(cfg) == cfg["tamanho_img"]:
        return mensagem
    return F.interpolate(mensagem, size=cfg["tamanho_img"], mode="nearest")


def agregar_logits(logits, cfg):
    if _tamanho_msg(cfg) == cfg["tamanho_img"]:
        return logits
    return F.adaptive_avg_pool2d(logits, _tamanho_msg(cfg))


@torch.no_grad()
def acuracia_bits(logits, mensagem):
    predito = (torch.sigmoid(logits) > 0.5).float()
    return (predito == mensagem).float().mean().item()


def _janela_gaussiana(tamanho, sigma, canais, dispositivo):
    coords = torch.arange(tamanho, device=dispositivo) - tamanho // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    janela2d = (g[:, None] * g[None, :])
    return janela2d.expand(canais, 1, tamanho, tamanho).contiguous()


@torch.no_grad()
def ssim(a, b, canais, tamanho_janela=11, sigma=1.5):

    janela = _janela_gaussiana(tamanho_janela, sigma, canais, a.device)
    pad = tamanho_janela // 2
    mu_a = F.conv2d(a, janela, padding=pad, groups=canais)
    mu_b = F.conv2d(b, janela, padding=pad, groups=canais)
    mu_a2, mu_b2, mu_ab = mu_a * mu_a, mu_b * mu_b, mu_a * mu_b
    var_a = F.conv2d(a * a, janela, padding=pad, groups=canais) - mu_a2
    var_b = F.conv2d(b * b, janela, padding=pad, groups=canais) - mu_b2
    cov_ab = F.conv2d(a * b, janela, padding=pad, groups=canais) - mu_ab
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    s = ((2 * mu_ab + c1) * (2 * cov_ab + c2)) / ((mu_a2 + mu_b2 + c1) * (var_a + var_b + c2))
    return s.mean().item()

def treinar():
    cfg = CONFIG
    torch.manual_seed(cfg["semente"]); random.seed(cfg["semente"])
    os.makedirs(cfg["pasta_saida"], exist_ok=True)
    dispositivo = cfg["dispositivo"]
    tm = _tamanho_msg(cfg)
    bpp = cfg["profundidade_dados"] * (tm / cfg["tamanho_img"]) ** 2
    print(f"[infra] dispositivo={dispositivo}  payload={bpp:.4g} bpp  "
          f"(tamanho_msg={tm}, tamanho_img={cfg['tamanho_img']})")

    loader_treino, loader_val = montar_loaders(cfg)

    codificador = Codificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    decodificador = Decodificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)


    otimizador = torch.optim.Adam(
        list(codificador.parameters()) + list(decodificador.parameters()), lr=cfg["taxa_aprend"])
    usar_amp = cfg["usar_amp"] and dispositivo == "cuda"
    escalador = torch.cuda.amp.GradScaler(enabled=usar_amp)

    melhor = {"score": float("-inf"), "epoca": 0, "acuracia_bits": 0.0, "psnr": 0.0, "ssim": 0.0}

    for epoca in range(1, cfg["epocas"] + 1):
        codificador.train(); decodificador.train()
        acumulado = {"img": 0.0, "msg": 0.0, "acc": 0.0, "n": 0}

        for portadora in loader_treino:
            portadora = portadora.to(dispositivo, non_blocking=True)
            mensagem = mensagem_aleatoria(portadora.size(0), cfg)
            mensagem_espalhada = espalhar_mensagem(mensagem, cfg)

            with torch.cuda.amp.autocast(enabled=usar_amp):
                estego = codificador(portadora, mensagem_espalhada)
                logits = agregar_logits(decodificador(estego), cfg)

                perda_imagem = F.mse_loss(estego, portadora)
                perda_mensagem = F.binary_cross_entropy_with_logits(logits, mensagem)
                perda_residuo = (estego - portadora).abs().mean()
                perda = (cfg["lambda_imagem"] * perda_imagem
                         + cfg["lambda_mensagem"] * perda_mensagem
                         + cfg["lambda_residuo"] * perda_residuo)


            otimizador.zero_grad()
            escalador.scale(perda).backward()
            escalador.step(otimizador)
            escalador.update()

            n_lote = portadora.size(0)
            acumulado["img"] += perda_imagem.item() * n_lote
            acumulado["msg"] += perda_mensagem.item() * n_lote
            acumulado["acc"] += acuracia_bits(logits, mensagem) * n_lote
            acumulado["n"] += n_lote

        n = acumulado["n"]
        print(f"[treino] época {epoca:3d}  img_mse={acumulado['img']/n:.5f}  "
              f"msg_bce={acumulado['msg']/n:.4f}  acuracia_bits={acumulado['acc']/n:.4f}")

       
        val = avaliar(codificador, decodificador, loader_val, cfg)
        ok = "OK" if (val["acuracia_bits"] >= cfg["alvo_bitacc"] and val["psnr"] >= 35.0) else "  "
        print(f"[  val ] época {epoca:3d}  acuracia_bits={val['acuracia_bits']:.4f}  "
              f"psnr={val['psnr']:.2f}dB  ssim={val['ssim']:.4f}  linf={val['linf']:.4f}  {ok}")

      
        score = (val["psnr"] if val["acuracia_bits"] >= cfg["alvo_bitacc"]
                 else val["acuracia_bits"] - 100.0)
        if score > melhor["score"]:
            melhor = {"score": score, "epoca": epoca,
                      **{k: val[k] for k in ("acuracia_bits", "psnr", "ssim")}}
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "epoca": epoca, "cfg": cfg, "val": val},
                       os.path.join(cfg["pasta_saida"], "ckpt_best.pt"))

        if epoca % cfg["salvar_cada"] == 0:
            salvar_amostras(codificador, loader_val, cfg, epoca)
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "epoca": epoca, "cfg": cfg},
                       os.path.join(cfg["pasta_saida"], "ckpt_last.pt"))

    print(f"\n[fim] checkpoints e amostras em: {cfg['pasta_saida']}")
    print(f"[melhor] época {melhor['epoca']}: acuracia_bits={melhor['acuracia_bits']:.4f}  "
          f"psnr={melhor['psnr']:.2f}dB  ssim={melhor['ssim']:.4f}")
    if melhor["acuracia_bits"] >= cfg["alvo_bitacc"] and melhor["psnr"] >= 35.0:
        print("[Fase 0 FECHADA] acurácia e PSNR>=35dB atingidos -> pode ligar a WGAN-GP (Fase 2).")
    else:
        print("[Fase 0 ainda aberta] baixar o payload (tamanho_msg menor) e/ou treinar mais antes da Fase 2.")
    return {"acuracia_bits": melhor["acuracia_bits"], "psnr": melhor["psnr"], "ssim": melhor["ssim"]}


@torch.no_grad()
def avaliar(codificador, decodificador, loader, cfg):
    codificador.eval(); decodificador.eval()
    acc, mse, ss, linf, n = 0.0, 0.0, 0.0, 0.0, 0
    for portadora in loader:
        portadora = portadora.to(cfg["dispositivo"])
        mensagem = mensagem_aleatoria(portadora.size(0), cfg)
        estego = codificador(portadora, espalhar_mensagem(mensagem, cfg))
        logits = agregar_logits(decodificador(estego), cfg)
        n_lote = portadora.size(0)
        acc += acuracia_bits(logits, mensagem) * n_lote
        mse += F.mse_loss(estego, portadora).item() * n_lote
        ss += ssim(estego, portadora, cfg["canais"]) * n_lote
        linf = max(linf, (estego - portadora).abs().max().item())
        n += n_lote
    mse = mse / max(n, 1)
    psnr = 10.0 * math.log10(1.0 / mse) if mse > 0 else float("inf")
    return {"acuracia_bits": acc / max(n, 1), "psnr": psnr, "ssim": ss / max(n, 1), "linf": linf}


@torch.no_grad()
def salvar_amostras(codificador, loader, cfg, epoca):
    codificador.eval()
    portadora = next(iter(loader)).to(cfg["dispositivo"])[:8]
    mensagem = mensagem_aleatoria(portadora.size(0), cfg)
    estego = codificador(portadora, espalhar_mensagem(mensagem, cfg))
    residuo = (estego - portadora).abs()
    residuo = residuo / (residuo.amax() + 1e-8)
    grade = torch.cat([portadora, estego, residuo], dim=0)
    caminho = os.path.join(cfg["pasta_saida"], f"samples_epoch{epoca:03d}.png")
    save_image(grade, caminho, nrow=8)


if __name__ == "__main__":
    treinar()


In [ ]:
%%writefile fase2_wgan_gp.py
import os
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

import fase0_codificador_decodificador as base


CONFIG = dict(base.CONFIG)
CONFIG.update({
    "lambda_adv": 0.01,       
                               
    "n_critic": 5,             
    "lambda_gp": 10.0,         
    "largura_critico": 64,     
    "lr_critico": 1e-4,       
    "beta1": 0.0, "beta2": 0.9,

    "taxa_aprend": 1e-4,      
    "epocas": 60,
    "usar_amp": False,
    "carregar_fase0": "runs/phase0_bossbase/ckpt_best.pt",
    "pasta_saida": "runs/phase2_bossbase",
})

COLAB_GPU = dict(base.COLAB_GPU)
COLAB_GPU.update({
    "epocas": 60,
    "lambda_adv": 0.01,
    "usar_amp": False,
})


class Critico(nn.Module):

    def __init__(self, canais, largura=64):
        super().__init__()

        def bloco(entrada, saida, norm=True):
            camadas = [nn.Conv2d(entrada, saida, 4, stride=2, padding=1)]
            if norm:
                camadas.append(nn.InstanceNorm2d(saida, affine=True))
            camadas.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*camadas)

        self.corpo = nn.Sequential(
            bloco(canais, largura, norm=False),
            bloco(largura, largura * 2),
            bloco(largura * 2, largura * 4),
            bloco(largura * 4, largura * 8),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.para_score = nn.Linear(largura * 8, 1)

    def forward(self, x):
        h = self.corpo(x)
        h = self.pool(h).flatten(1)
        return self.para_score(h).squeeze(1)


def penalidade_gradiente(critico, reais, falsas, dispositivo):

    b = reais.size(0)
    eps = torch.rand(b, 1, 1, 1, device=dispositivo)
    interp = (eps * reais + (1.0 - eps) * falsas).requires_grad_(True)
    score = critico(interp)
    grads = torch.autograd.grad(
        outputs=score, inputs=interp,
        grad_outputs=torch.ones_like(score),
        create_graph=True, retain_graph=True, only_inputs=True,
    )[0]
    grads = grads.reshape(b, -1)
    return ((grads.norm(2, dim=1) - 1.0) ** 2).mean()


def auc_binaria(scores, rotulos):
    scores = scores.detach().float().flatten()
    rotulos = rotulos.detach().float().flatten()
    n_pos = int(rotulos.sum().item())
    n_neg = int(rotulos.numel() - n_pos)
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    ordem = torch.argsort(scores)
    postos = torch.empty_like(scores)
    postos[ordem] = torch.arange(1, scores.numel() + 1, dtype=scores.dtype, device=scores.device)
    soma_postos_pos = postos[rotulos == 1].sum().item()
    return (soma_postos_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


@torch.no_grad()
def avaliar_critico(codificador, critico, loader, cfg):
    codificador.eval(); critico.eval()
    scores, rotulos = [], []
    for portadora in loader:
        portadora = portadora.to(cfg["dispositivo"])
        b = portadora.size(0)
        mensagem = base.mensagem_aleatoria(b, cfg)
        estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
        scores.append(critico(torch.cat([portadora, estego], dim=0)).cpu())
        rotulos.append(torch.cat([torch.zeros(b), torch.ones(b)]))
    codificador.train(); critico.train()
    return auc_binaria(torch.cat(scores), torch.cat(rotulos))


def treinar():
    cfg = CONFIG
    torch.manual_seed(cfg["semente"]); random.seed(cfg["semente"])
    os.makedirs(cfg["pasta_saida"], exist_ok=True)
    dispositivo = cfg["dispositivo"]
    tm = base._tamanho_msg(cfg)
    bpp = cfg["profundidade_dados"] * (tm / cfg["tamanho_img"]) ** 2
    print(f"[infra] dispositivo={dispositivo}  payload={bpp:.4g} bpp  "
          f"lambda_adv={cfg['lambda_adv']}  n_critic={cfg['n_critic']}")

    loader_treino, loader_val = base.montar_loaders(cfg)

    codificador = base.Codificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    decodificador = base.Decodificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    critico = Critico(cfg["canais"], cfg["largura_critico"]).to(dispositivo)

    ck = cfg.get("carregar_fase0")
    if ck and os.path.exists(ck):
        estado = torch.load(ck, map_location=dispositivo)
        codificador.load_state_dict(estado["codificador"])
        decodificador.load_state_dict(estado["decodificador"])
        print(f"[warm start] pesos da Fase 0 carregados de {ck}")
    else:
        print(f"[warm start] {ck!r} não encontrado -> treinando do zero "
              "(recomendado rodar a Fase 0 antes)")

    opt_g = torch.optim.Adam(list(codificador.parameters()) + list(decodificador.parameters()),
                             lr=cfg["taxa_aprend"], betas=(cfg["beta1"], cfg["beta2"]))
    opt_c = torch.optim.Adam(critico.parameters(),
                             lr=cfg["lr_critico"], betas=(cfg["beta1"], cfg["beta2"]))

    melhor = {"score": float("-inf"), "epoca": 0, "acuracia_bits": 0.0, "psnr": 0.0, "ssim": 0.0}

    for epoca in range(1, cfg["epocas"] + 1):
        codificador.train(); decodificador.train(); critico.train()
        ac = {"c": 0.0, "w": 0.0, "adv": 0.0, "img": 0.0, "msg": 0.0, "acc": 0.0, "n": 0}

        for portadora in loader_treino:
            portadora = portadora.to(dispositivo, non_blocking=True)
            b = portadora.size(0)

            for _ in range(cfg["n_critic"]):
                mensagem = base.mensagem_aleatoria(b, cfg)
                with torch.no_grad():
                    estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
                score_real = critico(portadora).mean()
                score_falsa = critico(estego).mean()
                gp = penalidade_gradiente(critico, portadora, estego, dispositivo)
                perda_c = score_falsa - score_real + cfg["lambda_gp"] * gp
                opt_c.zero_grad(); perda_c.backward(); opt_c.step()

            w_dist = (score_real - score_falsa).item()

            mensagem = base.mensagem_aleatoria(b, cfg)
            estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
            logits = base.agregar_logits(decodificador(estego), cfg)

            perda_img = F.mse_loss(estego, portadora)
            perda_msg = F.binary_cross_entropy_with_logits(logits, mensagem)
            perda_res = (estego - portadora).abs().mean()
            perda_adv = -critico(estego).mean()
            perda_g = (cfg["lambda_imagem"] * perda_img
                       + cfg["lambda_mensagem"] * perda_msg
                       + cfg["lambda_residuo"] * perda_res
                       + cfg["lambda_adv"] * perda_adv)
            opt_g.zero_grad(); perda_g.backward(); opt_g.step()

            ac["c"] += perda_c.item() * b;   ac["w"] += w_dist * b
            ac["adv"] += perda_adv.item() * b
            ac["img"] += perda_img.item() * b; ac["msg"] += perda_msg.item() * b
            ac["acc"] += base.acuracia_bits(logits, mensagem) * b
            ac["n"] += b

        n = ac["n"]
        print(f"[treino] época {epoca:3d}  perda_c={ac['c']/n:+.3f}  W={ac['w']/n:+.3f}  "
              f"adv={ac['adv']/n:+.3f}  img_mse={ac['img']/n:.5f}  "
              f"msg_bce={ac['msg']/n:.4f}  acuracia_bits={ac['acc']/n:.4f}")

        val = base.avaliar(codificador, decodificador, loader_val, cfg)
        auc_c = avaliar_critico(codificador, critico, loader_val, cfg)
        ok = "OK" if (val["acuracia_bits"] >= cfg["alvo_bitacc"] and val["psnr"] >= 35.0) else "  "
        print(f"[  val ] época {epoca:3d}  acuracia_bits={val['acuracia_bits']:.4f}  "
              f"psnr={val['psnr']:.2f}dB  ssim={val['ssim']:.4f}  linf={val['linf']:.4f}  "
              f"auc_critico={auc_c:.4f} (efetiva {max(auc_c, 1 - auc_c):.4f})  {ok}")

        score = (val["psnr"] if val["acuracia_bits"] >= cfg["alvo_bitacc"]
                 else val["acuracia_bits"] - 100.0)
        if score > melhor["score"]:
            melhor = {"score": score, "epoca": epoca,
                      **{k: val[k] for k in ("acuracia_bits", "psnr", "ssim")}}
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "critico": critico.state_dict(),
                        "epoca": epoca, "cfg": cfg, "val": val},
                       os.path.join(cfg["pasta_saida"], "ckpt_best.pt"))

        if epoca % cfg["salvar_cada"] == 0:
            base.salvar_amostras(codificador, loader_val, cfg, epoca)
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "critico": critico.state_dict(),
                        "epoca": epoca, "cfg": cfg},
                       os.path.join(cfg["pasta_saida"], "ckpt_last.pt"))

    print(f"\n[fim] checkpoints e amostras em: {cfg['pasta_saida']}")
    print(f"[melhor] época {melhor['epoca']}: acuracia_bits={melhor['acuracia_bits']:.4f}  "
          f"psnr={melhor['psnr']:.2f}dB  ssim={melhor['ssim']:.4f}")
    return {"acuracia_bits": melhor["acuracia_bits"], "psnr": melhor["psnr"], "ssim": melhor["ssim"]}


if __name__ == "__main__":
    treinar()


In [ ]:
%%writefile fase3_esteganalisador.py
import os
import csv
import time
import random
import shutil
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F

import fase0_codificador_decodificador as base
import fase2_wgan_gp as fase2
import fase25_residual as fase25


CONFIG = dict(fase2.CONFIG)
CONFIG.update({
    "arquitetura": "residual",

    "alvo_psnr": None,

    "lambda_srnet": 0.01,
    "n_srnet": 1,
    "alvo_srnet": "equilibrio",
    "lr_srnet": 1e-4,
    "srnet_compacta": False,

    "epocas": 60,
    "usar_amp": False,
    "carregar_fase0": None,
    "carregar_fase2": "runs/phase2_residual/ckpt_best.pt",

    "retomar_de": None,
    "salvar_otimizadores": True,
    "pasta_saida": "runs/phase3_bossbase",

    "espelho_ckpt": None,
})

COLAB_GPU = dict(fase2.COLAB_GPU)
COLAB_GPU.update({
    "epocas": 60,
    "lambda_srnet": 0.01,
    "usar_amp": False,
    "num_trabalhadores": max(1, min(4, (os.cpu_count() or 2) - 1)),
})


class _Tipo1(nn.Module):
    def __init__(self, entrada, saida):
        super().__init__()
        self.conv = nn.Conv2d(entrada, saida, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(saida)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)), inplace=True)


class _Tipo2(nn.Module):
    def __init__(self, canais):
        super().__init__()
        self.c1 = nn.Conv2d(canais, canais, 3, padding=1, bias=False)
        self.b1 = nn.BatchNorm2d(canais)
        self.c2 = nn.Conv2d(canais, canais, 3, padding=1, bias=False)
        self.b2 = nn.BatchNorm2d(canais)

    def forward(self, x):
        h = F.relu(self.b1(self.c1(x)), inplace=True)
        h = self.b2(self.c2(h))
        return x + h


class _Tipo3(nn.Module):
    def __init__(self, entrada, saida):
        super().__init__()
        self.c1 = nn.Conv2d(entrada, saida, 3, padding=1, bias=False)
        self.b1 = nn.BatchNorm2d(saida)
        self.c2 = nn.Conv2d(saida, saida, 3, padding=1, bias=False)
        self.b2 = nn.BatchNorm2d(saida)
        self.pool = nn.AvgPool2d(3, stride=2, padding=1)
        self.atalho_conv = nn.Conv2d(entrada, saida, 1, stride=2, bias=False)
        self.atalho_bn = nn.BatchNorm2d(saida)

    def forward(self, x):
        h = F.relu(self.b1(self.c1(x)), inplace=True)
        h = self.pool(self.b2(self.c2(h)))
        return self.atalho_bn(self.atalho_conv(x)) + h


class _Tipo4(nn.Module):
    def __init__(self, entrada, saida):
        super().__init__()
        self.c = nn.Conv2d(entrada, saida, 3, padding=1, bias=False)
        self.b = nn.BatchNorm2d(saida)

    def forward(self, x):
        h = F.relu(self.b(self.c(x)), inplace=True)
        return h.mean(dim=(2, 3))


class SRNet(nn.Module):
    def __init__(self, canais, compacta=False):
        super().__init__()
        e = 0.5 if compacta else 1.0
        f = lambda n: max(4, int(n * e))

        self.tipo1 = nn.Sequential(_Tipo1(canais, f(64)), _Tipo1(f(64), f(16)))
        self.tipo2 = nn.Sequential(*[_Tipo2(f(16)) for _ in range(5)])
        self.tipo3 = nn.Sequential(
            _Tipo3(f(16), f(16)),
            _Tipo3(f(16), f(64)),
            _Tipo3(f(64), f(128)),
            _Tipo3(f(128), f(256)),
        )
        self.tipo4 = _Tipo4(f(256), f(512))
        self.para_logit = nn.Linear(f(512), 1)

    def forward(self, x):
        h = self.tipo1(x)
        h = self.tipo2(h)
        h = self.tipo3(h)
        h = self.tipo4(h)
        return self.para_logit(h).squeeze(1)


auc_binaria = fase2.auc_binaria


@torch.no_grad()
def avaliar_srnet(codificador, srnet, loader, cfg):
    dispositivo = cfg["dispositivo"]
    codificador.eval(); srnet.eval()
    todos_scores, todos_rotulos = [], []

    for portadora in loader:
        portadora = portadora.to(dispositivo, non_blocking=True)
        b = portadora.size(0)
        mensagem = base.mensagem_aleatoria(b, cfg)
        estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))

        logits = srnet(torch.cat([portadora, estego], dim=0)).cpu()
        todos_scores.append(logits)
        todos_rotulos.append(torch.cat([torch.zeros(b), torch.ones(b)]))

    scores = torch.cat(todos_scores)
    rotulos = torch.cat(todos_rotulos)
    auc = auc_binaria(scores, rotulos)

    predito = (scores > 0).float()
    vp = float(((predito == 1) & (rotulos == 1)).sum())
    fp = float(((predito == 1) & (rotulos == 0)).sum())
    fn = float(((predito == 0) & (rotulos == 1)).sum())
    acc = (predito == rotulos).float().mean().item()
    precisao = vp / (vp + fp) if (vp + fp) > 0 else float("nan")
    recall = vp / (vp + fn) if (vp + fn) > 0 else float("nan")
    f1 = (2 * precisao * recall / (precisao + recall)
          if (precisao == precisao and recall == recall and precisao + recall > 0)
          else float("nan"))
    return {"auc": auc, "acuracia_deteccao": acc, "precisao": precisao,
            "recall": recall, "f1": f1}


def _carregar_gerador(caminho, codificador, decodificador, critico, dispositivo, log=print):
    if not caminho or not os.path.exists(caminho):
        log(f"[warm start] {caminho!r} não encontrado -> gerador DO ZERO.\n"
              "             A Fase 3 é para ser construída SOBRE a Fase 2. Rode a Fase 2 antes,\n"
              "             ou aponte 'carregar_fase2' para o ckpt_best.pt no Drive.")
        return None

    estado = torch.load(caminho, map_location=dispositivo)
    cod = estado.get("codificador", estado.get("encoder"))
    dec = estado.get("decodificador", estado.get("decoder"))
    if cod is None or dec is None:
        raise KeyError(f"{caminho}: não achei os pesos do codificador/decodificador. "
                       f"Chaves presentes: {list(estado.keys())}")
    try:
        codificador.load_state_dict(cod)
        decodificador.load_state_dict(dec)
    except RuntimeError as e:
        raise RuntimeError(f"{caminho}: os pesos não cabem no gerador '{fase25._ATUAL}'. "
                           "Confira cfg['arquitetura'] - checkpoint denso só carrega em gerador "
                           "denso, residual só em residual.") from e

    if "critico" in estado:
        critico.load_state_dict(estado["critico"])
        log(f"[warm start] gerador + crítico carregados de {caminho} (época {estado.get('epoca', '?')})")
    else:
        log(f"[warm start] gerador carregado de {caminho}; crítico DO ZERO "
              "(checkpoint não tinha os pesos do crítico)")
    return estado.get("val", {}).get("psnr")


class _Log:
    def __init__(self, caminho):
        self.caminho = caminho
        self.arquivo = open(caminho, "a", encoding="utf-8")

    def __call__(self, texto=""):
        print(texto)
        self.arquivo.write(texto + "\n")
        self.arquivo.flush()

    def fechar(self):
        self.arquivo.close()


def _espelhar(cfg, origem, log=print):
    destino = cfg.get("espelho_ckpt")
    if not destino:
        return
    if os.path.abspath(destino) == os.path.abspath(cfg["pasta_saida"]):
        return
    try:
        os.makedirs(destino, exist_ok=True)
        shutil.copy2(origem, os.path.join(destino, os.path.basename(origem)))
    except Exception as e:
        log(f"[espelho] falhou ao copiar para {destino}: {e}")


def treinar():
    cfg = CONFIG
    torch.manual_seed(cfg["semente"]); random.seed(cfg["semente"])
    os.makedirs(cfg["pasta_saida"], exist_ok=True)
    dispositivo = cfg["dispositivo"]
    tm = base._tamanho_msg(cfg)
    bpp = cfg["profundidade_dados"] * (tm / cfg["tamanho_img"]) ** 2

    log = _Log(os.path.join(cfg["pasta_saida"], "treino.log"))
    log(f"===== Fase 3 iniciada em {datetime.now():%Y-%m-%d %H:%M:%S} =====")
    log("[cfg] " + "  ".join(f"{k}={cfg[k]}" for k in
        ("tamanho_img", "tamanho_msg", "tamanho_lote", "largura", "epocas",
         "lambda_imagem", "lambda_mensagem", "lambda_residuo", "lambda_adv",
         "lambda_srnet", "n_critic", "n_srnet", "lr_srnet", "srnet_compacta", "arquitetura")))
    log(f"[infra] dispositivo={dispositivo}  payload={bpp:.4g} bpp  "
          f"lambda_adv={cfg['lambda_adv']}  lambda_srnet={cfg['lambda_srnet']}  "
          f"n_critic={cfg['n_critic']}  n_srnet={cfg['n_srnet']}"
          + ("  [SRNet COMPACTA]" if cfg["srnet_compacta"] else ""))

    loader_treino, loader_val = base.montar_loaders(cfg)

    fase25.usar(cfg["arquitetura"])
    codificador = base.Codificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    decodificador = base.Decodificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    critico = fase2.Critico(cfg["canais"], cfg["largura_critico"]).to(dispositivo)
    srnet = SRNet(cfg["canais"], compacta=cfg["srnet_compacta"]).to(dispositivo)

    psnr_partida = _carregar_gerador(cfg.get("carregar_fase2"), codificador, decodificador,
                                     critico, dispositivo, log=log)
    if cfg["alvo_psnr"] is not None:
        alvo_psnr = cfg["alvo_psnr"]
        log(f"[alvo  ] critério (2): PSNR >= {alvo_psnr:.2f} dB (fixado em cfg)")
    elif psnr_partida is not None:
        alvo_psnr = round(psnr_partida - 1.0, 2)
        log(f"[alvo  ] critério (2): PSNR >= {alvo_psnr:.2f} dB (partida {psnr_partida:.2f} - 1 dB)")
    else:
        alvo_psnr = 37.27
        log(f"[alvo  ] checkpoint sem PSNR gravado -> critério (2) no padrão da Fase 2.5: "
            f"PSNR >= {alvo_psnr:.2f} dB")
    n_par = sum(p.numel() for p in srnet.parameters())
    log(f"[srnet] iniciada DO ZERO - {n_par/1e6:.2f}M parâmetros")

    opt_g = torch.optim.Adam(list(codificador.parameters()) + list(decodificador.parameters()),
                             lr=cfg["taxa_aprend"], betas=(cfg["beta1"], cfg["beta2"]))
    opt_c = torch.optim.Adam(critico.parameters(),
                             lr=cfg["lr_critico"], betas=(cfg["beta1"], cfg["beta2"]))
    opt_s = torch.optim.Adam(srnet.parameters(), lr=cfg["lr_srnet"])

    melhor = {"score": float("-inf"), "epoca": 0, "acuracia_bits": 0.0, "psnr": 0.0,
              "ssim": 0.0, "auc": float("nan")}
    epoca_inicial = 1
    ck_ret = cfg.get("retomar_de")
    if ck_ret and os.path.exists(ck_ret):
        est = torch.load(ck_ret, map_location=dispositivo)
        codificador.load_state_dict(est["codificador"])
        decodificador.load_state_dict(est["decodificador"])
        critico.load_state_dict(est["critico"])
        srnet.load_state_dict(est["srnet"])
        if "opt_g" in est:
            opt_g.load_state_dict(est["opt_g"])
            opt_c.load_state_dict(est["opt_c"])
            opt_s.load_state_dict(est["opt_s"])
        else:
            log("[retomada] o checkpoint não tem o estado dos otimizadores: o Adam recomeça "
                "do zero e as primeiras épocas dão um solavanco.")
        if est.get("melhor"):
            melhor = est["melhor"]
        if est.get("rng") is not None:
            torch.set_rng_state(est["rng"].cpu())
        epoca_inicial = int(est["epoca"]) + 1
        log(f"[retomada] {ck_ret} - continuando da época {epoca_inicial} "
            f"(melhor até aqui: época {melhor['epoca']}, PSNR {melhor['psnr']:.2f} dB)")
        if epoca_inicial > cfg["epocas"]:
            log(f"[retomada] o run já tinha {est['epoca']} épocas e cfg['epocas'] = "
                f"{cfg['epocas']}. Suba 'epocas' para continuar.")
    elif ck_ret:
        log(f"[retomada] {ck_ret!r} não encontrado -> começando do zero.")

    retomando = epoca_inicial > 1
    caminho_csv = os.path.join(cfg["pasta_saida"], "metricas_val.csv")
    if not (retomando and os.path.exists(caminho_csv)):
        with open(caminho_csv, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(["epoca", "acuracia_bits", "psnr", "ssim", "auc_srnet",
                                    "auc_critico", "w_dist", "acuracia_det", "precisao",
                                    "recall", "f1"])

    caminho_csv_treino = os.path.join(cfg["pasta_saida"], "metricas_treino.csv")
    if not (retomando and os.path.exists(caminho_csv_treino)):
        with open(caminho_csv_treino, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(["epoca", "perda_c", "w_dist", "adv", "srnet_bce", "enganar",
                                    "img_mse", "msg_bce", "acuracia_bits"])

    t_inicio = time.time()
    for epoca in range(epoca_inicial, cfg["epocas"] + 1):
        t_epoca = time.time()
        codificador.train(); decodificador.train(); critico.train(); srnet.train()
        ac = {"c": 0.0, "w": 0.0, "adv": 0.0, "s": 0.0, "eng": 0.0,
              "img": 0.0, "msg": 0.0, "acc": 0.0, "n": 0}

        for portadora in loader_treino:
            portadora = portadora.to(dispositivo, non_blocking=True)
            b = portadora.size(0)

            for _ in range(cfg["n_critic"]):
                mensagem = base.mensagem_aleatoria(b, cfg)
                with torch.no_grad():
                    estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
                score_real = critico(portadora).mean()
                score_falsa = critico(estego).mean()
                gp = fase2.penalidade_gradiente(critico, portadora, estego, dispositivo)
                perda_c = score_falsa - score_real + cfg["lambda_gp"] * gp
                opt_c.zero_grad(); perda_c.backward(); opt_c.step()

            w_dist = (score_real - score_falsa).item()

            for _ in range(cfg["n_srnet"]):
                mensagem = base.mensagem_aleatoria(b, cfg)
                with torch.no_grad():
                    estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
                logits_s = srnet(torch.cat([portadora, estego], dim=0))
                alvos_s = torch.cat([torch.zeros(b, device=dispositivo),
                                     torch.ones(b, device=dispositivo)])
                perda_s = F.binary_cross_entropy_with_logits(logits_s, alvos_s)
                opt_s.zero_grad(); perda_s.backward(); opt_s.step()

            mensagem = base.mensagem_aleatoria(b, cfg)
            estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
            logits = base.agregar_logits(decodificador(estego), cfg)

            perda_img = F.mse_loss(estego, portadora)
            perda_msg = F.binary_cross_entropy_with_logits(logits, mensagem)
            perda_res = (estego - portadora).abs().mean()
            perda_adv = -critico(estego).mean()

            logits_g = srnet(torch.cat([portadora.detach(), estego], dim=0))
            logit_est_g = logits_g[b:]
            valor_alvo = 0.5 if cfg["alvo_srnet"] == "equilibrio" else 0.0
            perda_eng = F.binary_cross_entropy_with_logits(
                logit_est_g, torch.full_like(logit_est_g, valor_alvo))

            perda_g = (cfg["lambda_imagem"] * perda_img
                       + cfg["lambda_mensagem"] * perda_msg
                       + cfg["lambda_residuo"] * perda_res
                       + cfg["lambda_adv"] * perda_adv
                       + cfg["lambda_srnet"] * perda_eng)
            opt_g.zero_grad(); perda_g.backward(); opt_g.step()

            ac["c"] += perda_c.item() * b;    ac["w"] += w_dist * b
            ac["adv"] += perda_adv.item() * b; ac["s"] += perda_s.item() * b
            ac["eng"] += perda_eng.item() * b
            ac["img"] += perda_img.item() * b; ac["msg"] += perda_msg.item() * b
            ac["acc"] += base.acuracia_bits(logits, mensagem) * b
            ac["n"] += b

        n = ac["n"]
        seg = time.time() - t_epoca
        feitas = epoca - epoca_inicial + 1
        restam = (cfg["epocas"] - epoca) * (time.time() - t_inicio) / feitas
        log(f"[tempo ] época {epoca:3d}  {seg:6.1f}s  ({ac['n']//cfg['tamanho_lote']} passos de gerador)"
            f"  restam ~{restam/60:.0f} min")
        log(f"[treino] época {epoca:3d}  perda_c={ac['c']/n:+.3f}  W={ac['w']/n:+.3f}  "
              f"srnet_bce={ac['s']/n:.4f}  enganar={ac['eng']/n:.4f}  "
              f"img_mse={ac['img']/n:.5f}  msg_bce={ac['msg']/n:.4f}  "
              f"acuracia_bits={ac['acc']/n:.4f}")

        val = base.avaliar(codificador, decodificador, loader_val, cfg)
        det = avaliar_srnet(codificador, srnet, loader_val, cfg)
        auc, auc_ef = det["auc"], max(det["auc"], 1.0 - det["auc"])
        auc_c = fase2.avaliar_critico(codificador, critico, loader_val, cfg)

        c1 = val["acuracia_bits"] >= cfg["alvo_bitacc"]
        c2 = val["psnr"] >= alvo_psnr
        c3 = abs(auc - 0.5) <= 0.05
        marca = "".join(["1" if c1 else ".", "2" if c2 else ".", "3" if c3 else "."])
        log(f"[  val ] época {epoca:3d}  acuracia_bits={val['acuracia_bits']:.4f}  "
              f"psnr={val['psnr']:.2f}dB  ssim={val['ssim']:.4f}  "
              f"auc_srnet={auc:.4f} (efetiva {auc_ef:.4f})  "
              f"auc_critico={auc_c:.4f} (efetiva {max(auc_c, 1 - auc_c):.4f})  "
              f"criterios[{marca}]")
        log(f"[detect] época {epoca:3d}  acuracia={det['acuracia_deteccao']:.4f}  "
            f"precisao={det['precisao']:.4f}  recall={det['recall']:.4f}  "
            f"f1={det['f1']:.4f}")

        with open(caminho_csv, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([epoca, f"{val['acuracia_bits']:.4f}", f"{val['psnr']:.2f}",
                                    f"{val['ssim']:.4f}", f"{auc:.4f}", f"{auc_c:.4f}",
                                    f"{ac['w']/n:.4f}", f"{det['acuracia_deteccao']:.4f}",
                                    f"{det['precisao']:.4f}", f"{det['recall']:.4f}",
                                    f"{det['f1']:.4f}"])
        with open(caminho_csv_treino, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([epoca, f"{ac['c']/n:.4f}", f"{ac['w']/n:.4f}",
                                    f"{ac['adv']/n:.4f}", f"{ac['s']/n:.4f}", f"{ac['eng']/n:.4f}",
                                    f"{ac['img']/n:.6f}", f"{ac['msg']/n:.4f}", f"{ac['acc']/n:.4f}"])

        for reg in (caminho_csv, caminho_csv_treino, os.path.join(cfg["pasta_saida"], "treino.log")):
            _espelhar(cfg, reg, log)

        score = (val["psnr"] if c1 else val["acuracia_bits"] - 100.0)
        if score > melhor["score"]:
            melhor = {"score": score, "epoca": epoca, "auc": auc, "det": det,
                      **{k: val[k] for k in ("acuracia_bits", "psnr", "ssim")}}
            destino = os.path.join(cfg["pasta_saida"], "ckpt_best.pt")
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "critico": critico.state_dict(),
                        "srnet": srnet.state_dict(),
                        "epoca": epoca, "cfg": cfg, "val": val, "det": det},
                       destino)
            _espelhar(cfg, destino, log)

        if epoca % cfg["salvar_cada"] == 0:
            base.salvar_amostras(codificador, loader_val, cfg, epoca)
            destino = os.path.join(cfg["pasta_saida"], "ckpt_last.pt")
            estado = {"codificador": codificador.state_dict(),
                      "decodificador": decodificador.state_dict(),
                      "critico": critico.state_dict(),
                      "srnet": srnet.state_dict(),
                      "epoca": epoca, "cfg": cfg,
                      "melhor": melhor, "rng": torch.get_rng_state()}
            if cfg.get("salvar_otimizadores", True):
                estado.update(opt_g=opt_g.state_dict(), opt_c=opt_c.state_dict(),
                              opt_s=opt_s.state_dict())
            torch.save(estado, destino)
            _espelhar(cfg, destino, log)

    auc_ef = max(melhor["auc"], 1.0 - melhor["auc"])
    log(f"\n[fim] checkpoints, amostras e metricas_val.csv em: {cfg['pasta_saida']}")
    log(f"[melhor] época {melhor['epoca']}: acuracia_bits={melhor['acuracia_bits']:.4f}  "
          f"psnr={melhor['psnr']:.2f}dB  ssim={melhor['ssim']:.4f}  auc_srnet={melhor['auc']:.4f}")
    log("\nCRITÉRIOS DE SAÍDA DA FASE 3:")
    log(f"  (1) acuracia_bits >= 0.99          -> {melhor['acuracia_bits']:.4f}  "
          f"{'OK' if melhor['acuracia_bits'] >= 0.99 else 'FALHOU: baixe lambda_srnet'}")
    log(f"  (2) PSNR >= {alvo_psnr:.2f} dB (partida - 1dB) -> {melhor['psnr']:.2f}  "
          f"{'OK' if melhor['psnr'] >= alvo_psnr else 'FALHOU: baixe lambda_srnet ou suba lambda_imagem'}")
    if melhor["acuracia_bits"] < cfg["alvo_bitacc"]:
        veredito = ("SEM VALOR: a mensagem não é recuperada - o gerador engana o detector "
                    "deixando de esconder. Baixe lambda_srnet")
    elif abs(melhor["auc"] - 0.5) <= 0.05:
        veredito = "OK"
    elif melhor["auc"] > 0.5:
        veredito = "detector separando: suba lambda_srnet"
    else:
        veredito = ("gerador PASSOU DO PONTO (estego com score menor que portadora). "
                    "NÃO suba lambda_srnet - isso piora. Use alvo_srnet='equilibrio'")
    log(f"  (3) AUC do laço ~ 0.5              -> {melhor['auc']:.4f} (efetiva {auc_ef:.4f})  {veredito}")
    d = melhor.get("det") or {}
    if d:
        log(f"      detector na melhor época: acuracia={d['acuracia_deteccao']:.4f}  "
            f"precisao={d['precisao']:.4f}  recall={d['recall']:.4f}  f1={d['f1']:.4f}")
        if d["recall"] > 0.95 and d["precisao"] < 0.55:
            log("      ATENÇÃO: recall ~1 com precisão ~0,5 = o detector responde 'estego' "
                "para tudo. Isso é detector que NAO CONVERGIU, nao seguranca.")
    log("\nATENÇÃO: o critério (3) mede o detector que participou do treino. Ele NÃO prova")
    log("segurança - pode significar que aquela SRNet não convergiu. A prova é a Fase 4,")
    log("que treina um esteganalisador independente e mede P_E. Ver ROADMAP.md.")
    log(f"\n[registros] treino.log, metricas_val.csv e metricas_treino.csv em {cfg['pasta_saida']}")
    log(f"[retomada] se a sessão cair, rode de novo com "
        f"cfg['retomar_de'] = '{os.path.join(cfg['pasta_saida'], 'ckpt_last.pt')}'")
    if not cfg.get("espelho_ckpt"):
        log("\nLEMBRETE: 'espelho_ckpt' está vazio. Se isto rodou no Colab, BAIXE AGORA o")
        log("ckpt_best.pt E os arquivos de registro - foi assim que os pesos das Fases 0 e 2")
        log("se perderam, e a saída da célula do notebook some junto com o runtime.")
    else:
        _espelhar(cfg, os.path.join(cfg["pasta_saida"], "treino.log"), log)
        log(f"[espelho] registros e checkpoints copiados para {cfg['espelho_ckpt']}")
    log.fechar()
    return {"acuracia_bits": melhor["acuracia_bits"], "psnr": melhor["psnr"],
            "ssim": melhor["ssim"], "auc": melhor["auc"], "alvo_psnr": alvo_psnr}


if __name__ == "__main__":
    treinar()


In [ ]:
%%writefile fase25_residual.py
import os
import shutil
import sys
import time

import torch
import torch.nn as nn

import fase0_codificador_decodificador as base
import fase2_wgan_gp as fase2


class BlocoResidual(nn.Module):

    def __init__(self, canais):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(canais, canais, 3, padding=1),
            nn.BatchNorm2d(canais),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(canais, canais, 3, padding=1),
            nn.BatchNorm2d(canais),
        )

    def forward(self, x):
        return x + self.conv(x)


class CodificadorResidual(nn.Module):
  

    def __init__(self, canais, profundidade_dados, largura=32, blocos=3):
        super().__init__()
        self.features = base.bloco_conv(canais, largura)
        self.entrada = base.bloco_conv(largura + profundidade_dados, largura)
        self.residuais = nn.Sequential(*[BlocoResidual(largura) for _ in range(blocos)])
        self.para_residuo = nn.Conv2d(largura, canais, 3, padding=1)

    def forward(self, portadora, mensagem):
        x = self.features(portadora)
        x = self.entrada(torch.cat([x, mensagem], dim=1))
        x = self.residuais(x)
        residuo = self.para_residuo(x)
        return (portadora + residuo).clamp(0.0, 1.0)


class DecodificadorResidual(nn.Module):

    def __init__(self, canais, profundidade_dados, largura=32, blocos=3):
        super().__init__()
        self.entrada = base.bloco_conv(canais, largura)
        self.residuais = nn.Sequential(*[BlocoResidual(largura) for _ in range(blocos)])
        self.para_logits = nn.Conv2d(largura, profundidade_dados, 3, padding=1)

    def forward(self, estego):
        x = self.entrada(estego)
        x = self.residuais(x)
        return self.para_logits(x)


_ORIGINAIS = {"Codificador": base.Codificador, "Decodificador": base.Decodificador}
_ATUAL = "denso"

_EXTRAS = {}


def registrar(nome, fabrica_cod, fabrica_dec=None):
    _EXTRAS[nome] = (fabrica_cod, fabrica_dec)
    return nome

BLOCOS_RESIDUAIS = 3


def usar(arquitetura):

    global _ATUAL
    if arquitetura in _EXTRAS:
        fab_cod, fab_dec = _EXTRAS[arquitetura]
        base.Codificador = fab_cod
        base.Decodificador = fab_dec or (
            lambda c, p, l=32: DecodificadorResidual(c, p, l, BLOCOS_RESIDUAIS))
        _ATUAL = arquitetura
        print(f"[arquitetura] gerador = {arquitetura}")
        return arquitetura

    if arquitetura not in ("denso", "residual"):
        raise ValueError('arquitetura deve ser "denso", "residual" ou uma registrada: '
                         + ", ".join(sorted(_EXTRAS)) or "(nenhuma)")

    if arquitetura == "denso":
        base.Codificador = _ORIGINAIS["Codificador"]
        base.Decodificador = _ORIGINAIS["Decodificador"]
    else:
        def _cod(canais, profundidade, largura=32):
            return CodificadorResidual(canais, profundidade, largura, BLOCOS_RESIDUAIS)

        def _dec(canais, profundidade, largura=32):
            return DecodificadorResidual(canais, profundidade, largura, BLOCOS_RESIDUAIS)

        base.Codificador = _cod
        base.Decodificador = _dec

    _ATUAL = arquitetura
    print(f"[arquitetura] gerador = {arquitetura}")
    return arquitetura


def contar_parametros(canais=1, profundidade=1, largura=64):
 
    resultado = {}
    for nome, (cod, dec) in {
        "denso": (_ORIGINAIS["Codificador"], _ORIGINAIS["Decodificador"]),
        "residual": (
            lambda c, p, l: CodificadorResidual(c, p, l, BLOCOS_RESIDUAIS),
            lambda c, p, l: DecodificadorResidual(c, p, l, BLOCOS_RESIDUAIS),
        ),
    }.items():
        c = cod(canais, profundidade, largura)
        d = dec(canais, profundidade, largura)
        resultado[nome] = {
            "codificador": sum(p.numel() for p in c.parameters()),
            "decodificador": sum(p.numel() for p in d.parameters()),
        }
        resultado[nome]["total"] = (resultado[nome]["codificador"]
                                    + resultado[nome]["decodificador"])
    return resultado


class _Tee:

    def __init__(self, caminho):
        self.caminho = caminho
        self.arquivo = None
        self.stdout = None

    def __enter__(self):
        os.makedirs(os.path.dirname(self.caminho) or ".", exist_ok=True)
        self.arquivo = open(self.caminho, "a", encoding="utf-8")
        self.stdout = sys.stdout
        sys.stdout = self
        return self

    def write(self, texto):
        self.stdout.write(texto)
        self.arquivo.write(texto)
        self.arquivo.flush()

    def flush(self):
        self.stdout.flush()
        self.arquivo.flush()

    def __exit__(self, *exc):
        sys.stdout = self.stdout
        self.arquivo.close()


def _gerar_csvs(log, pasta):
    try:
        from logs_para_csv import ler_log, gravar
    except ImportError:
        print("[csv] logs_para_csv.py não encontrado; CSVs não gerados")
        return
    treino, val = ler_log(log)
    for linhas, nome in ((treino, "metricas_treino.csv"), (val, "metricas_val.csv")):
        if linhas:
            gravar(linhas, os.path.join(pasta, nome))
            print(f"[csv] {nome}: {len(linhas)} linhas")


def espelhar(pasta, destino):
    if not destino or os.path.abspath(destino) == os.path.abspath(pasta):
        return
    os.makedirs(destino, exist_ok=True)
    copiados = 0
    for nome in os.listdir(pasta):
        origem = os.path.join(pasta, nome)
        if os.path.isfile(origem):
            shutil.copy2(origem, os.path.join(destino, nome))
            copiados += 1
    print(f"[espelho] {copiados} arquivo(s) copiado(s) para {destino}")


def _rodar(modulo, cfg_extra, pasta_saida, espelho, rotulo):
    rotulo = f"{rotulo} ({_ATUAL})"
    modulo.CONFIG.update(base.COLAB_GPU)
    modulo.CONFIG.update(cfg_extra)
    modulo.CONFIG["pasta_saida"] = pasta_saida
    os.makedirs(pasta_saida, exist_ok=True)
    log = os.path.join(pasta_saida, "treino.log")

    inicio = time.time()
    with _Tee(log):
        print(f"\n===== {rotulo} iniciado em "
              f"{time.strftime('%Y-%m-%d %H:%M:%S')} =====")
        print(f"[arquitetura] {_ATUAL}  |  blocos residuais = {BLOCOS_RESIDUAIS}")
        print(f"[cfg] " + "  ".join(
            f"{k}={modulo.CONFIG[k]}" for k in
            ("tamanho_img", "tamanho_msg", "tamanho_lote", "largura", "epocas",
             "lambda_imagem", "lambda_residuo") if k in modulo.CONFIG))
        resultado = modulo.treinar()
        minutos = (time.time() - inicio) / 60
        print(f"[tempo ] {rotulo}: {minutos:.1f} min "
              f"({minutos * 60 / max(modulo.CONFIG['epocas'], 1):.1f} s/época)")

    _gerar_csvs(log, pasta_saida)
    espelhar(pasta_saida, espelho)
    return resultado


def treinar_fase0(pasta_saida, espelho=None, **cfg_extra):
    return _rodar(base, cfg_extra, pasta_saida, espelho, "Fase 0")


def treinar_fase2(pasta_saida, espelho=None, **cfg_extra):
    return _rodar(fase2, cfg_extra, pasta_saida, espelho, "Fase 2")


ALVO_ACURACIA = 0.99
ALVO_PSNR_FASE2 = 36.30


def _ler_csv(caminho):
    import csv
    if not os.path.exists(caminho):
        return []
    with open(caminho, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def comparar(csv_denso, csv_residual, alvo_psnr=ALVO_PSNR_FASE2, rotulo="Fase 2"):
 
    d, r = _ler_csv(csv_denso), _ler_csv(csv_residual)
    if not r:
        print(f"[comparar] {csv_residual} não encontrado - rode o treino antes.")
        return None
    if not d:
        print(f"[comparar] {csv_denso} não encontrado - sem linha de base para comparar.")
        return None

    n = min(len(d), len(r))
    print(f"\n{'=' * 74}\n{rotulo}: DENSO × RESIDUAL nas primeiras {n} épocas\n{'=' * 74}")

    def resumo(linhas, nome):
        linhas = linhas[:n]
        psnr = [float(x["psnr"]) for x in linhas if x.get("psnr")]
        acc = [float(x["acuracia_bits"]) for x in linhas if x.get("acuracia_bits")]
        ssim = [float(x["ssim"]) for x in linhas if x.get("ssim")]
        melhor = max(psnr) if psnr else 0.0
        ult10 = psnr[-10:] or [0.0]
        print(f"  {nome:<9} melhor PSNR {melhor:6.2f} dB | "
              f"média 10 últimas {sum(ult10) / len(ult10):6.2f} dB | "
              f"acurácia mín {min(acc) if acc else 0:.4f} | "
              f"SSIM máx {max(ssim) if ssim else 0:.4f} | "
              f"épocas >= {alvo_psnr:.2f} dB: {sum(1 for p in psnr if p >= alvo_psnr)}")
        return {"melhor": melhor, "acc_min": min(acc) if acc else 0.0,
                "media10": sum(ult10) / len(ult10)}

    rd, rr = resumo(d, "denso"), resumo(r, "residual")

    print(f"\n  diferença (residual - denso): "
          f"{rr['melhor'] - rd['melhor']:+.2f} dB no melhor, "
          f"{rr['media10'] - rd['media10']:+.2f} dB na média das 10 últimas")

    print(f"\n{'-' * 74}\nCRITÉRIOS DE SAÍDA DA FASE 2.5")
    c1 = rr["acc_min"] >= ALVO_ACURACIA
    c2 = rr["melhor"] >= alvo_psnr
    print(f"  (1) acurácia de bits >= {ALVO_ACURACIA}      -> {rr['acc_min']:.4f}  "
          f"{'OK' if c1 else 'FALHOU'}")
    print(f"  (2) PSNR >= {alvo_psnr:.2f} dB (não regredir) -> {rr['melhor']:.2f}  "
          f"{'OK' if c2 else 'FALHOU'}")
    print("  (3) custo por época medido            -> ver a linha [tempo ] em cada treino.log")
    print("  (4) os dois geradores preservados     -> confira o espelho no Drive")

    if c1 and c2:
        print("\n  Residual passa. Decisão de qual segue: maior PSNR a igual acurácia;")
        print("  empate técnico (< 0,2 dB) fica com o mais barato por época.")
    else:
        print("\n  Residual NÃO bateu os critérios. Resultado negativo também é resultado:")
        print("  registre no ROADMAP e siga com o denso.")
    return {"denso": rd, "residual": rr}


In [ ]:
%%writefile fase6_adaptativo.py
import torch
import torch.nn as nn
import torch.nn.functional as F

import fase0_codificador_decodificador as base
import fase25_residual as f25

JANELA = 5
PISO = 0.25
TETO = 3.00
BLOCO = 4
NORMALIZACAO = "global"
PRESERVAR = "media"


def textura_local(x, janela=None):
    janela = janela or JANELA
    pad = janela // 2
    media = F.avg_pool2d(x, janela, stride=1, padding=pad, count_include_pad=False)
    media_quad = F.avg_pool2d(x * x, janela, stride=1, padding=pad, count_include_pad=False)
    return (media_quad - media * media).clamp_min(0.0).sqrt()


def _media(t, bloco, modo):
    if modo == "bloco":
        m = F.avg_pool2d(t, bloco, stride=bloco)
        return F.interpolate(m, size=t.shape[-2:], mode="nearest")
    return t.mean(dim=(1, 2, 3), keepdim=True).expand_as(t)


def mascara_textura(x, bloco=None, janela=None, piso=None, teto=None, modo=None,
                    preservar=None):
    bloco = bloco or BLOCO
    piso = PISO if piso is None else piso
    teto = TETO if teto is None else teto
    modo = modo or NORMALIZACAO
    preservar = preservar or PRESERVAR

    t = textura_local(x, janela)
    m = (t / (_media(t, bloco, modo) + 1e-6)).clamp(piso, teto)
    if preservar == "energia":
        return m / (_media(m * m, bloco, modo) + 1e-6).sqrt()
    return m / (_media(m, bloco, modo) + 1e-6)


class CodificadorAdaptativo(nn.Module):

    def __init__(self, canais, profundidade_dados, largura=32, blocos=3):
        super().__init__()
        self.nucleo = f25.CodificadorResidual(canais, profundidade_dados, largura, blocos)

    def load_state_dict(self, estado, strict=True):
        if estado and not any(k.startswith("nucleo.") for k in estado):
            estado = {f"nucleo.{k}": v for k, v in estado.items()}
        return super().load_state_dict(estado, strict=strict)

    def forward(self, portadora, mensagem):
        x = self.nucleo.features(portadora)
        x = self.nucleo.entrada(torch.cat([x, mensagem], dim=1))
        x = self.nucleo.residuais(x)
        residuo = self.nucleo.para_residuo(x)
        with torch.no_grad():
            m = mascara_textura(portadora)
        return (portadora + residuo * m).clamp(0.0, 1.0)


def _fabrica_codificador(canais, profundidade, largura=32):
    return CodificadorAdaptativo(canais, profundidade, largura, f25.BLOCOS_RESIDUAIS)


f25.registrar("adaptativo", _fabrica_codificador)

_ATUAL = "residual"


def usar(arquitetura, bloco=None, janela=None, piso=None, teto=None, normalizacao=None,
         preservar=None):
    global _ATUAL, BLOCO, JANELA, PISO, TETO, NORMALIZACAO, PRESERVAR
    if preservar is not None:
        if preservar not in ("media", "energia"):
            raise ValueError('preservar deve ser "media" ou "energia"')
        PRESERVAR = preservar
    if normalizacao is not None:
        if normalizacao not in ("bloco", "global"):
            raise ValueError('normalizacao deve ser "bloco" ou "global"')
        NORMALIZACAO = normalizacao
    if bloco is not None:
        BLOCO = int(bloco)
    if janela is not None:
        JANELA = int(janela)
    if piso is not None:
        PISO = float(piso)
    if teto is not None:
        TETO = float(teto)

    if arquitetura != "adaptativo":
        f25.usar(arquitetura)
        _ATUAL = arquitetura
        return arquitetura

    f25.usar("adaptativo")
    _ATUAL = "adaptativo"
    print(f"[arquitetura] gerador = adaptativo  |  normalizacao = {NORMALIZACAO}  "
          f"preservar = {PRESERVAR}  bloco = {BLOCO}  janela = {JANELA}  piso = {PISO}  teto = {TETO}")
    return arquitetura


def carregar_nucleo(caminho, codificador, dispositivo="cpu"):
    estado = torch.load(caminho, map_location=dispositivo)
    cod = estado.get("codificador", estado.get("encoder"))
    codificador.nucleo.load_state_dict(cod)
    return estado.get("epoca", "?")


def _autoteste():
    torch.manual_seed(0)
    x = torch.zeros(2, 1, 32, 32)
    x[..., 16:] = torch.rand(2, 1, 32, 16)
    for modo in ("bloco", "global"):
        for preservar in ("media", "energia"):
            m = mascara_textura(x, bloco=4, modo=modo, preservar=preservar)
            print(f"[autoteste] {modo:6s} {preservar:7s} | E[m] {m.mean():.4f}  "
                  f"E[m²] {(m * m).mean():.4f} | liso {m[..., :16].mean():.4f}  "
                  f"texturizado {m[..., 16:].mean():.4f} | faixa {m.min():.3f}-{m.max():.3f}")

    par_res = sum(p.numel() for p in f25.CodificadorResidual(1, 1, 64, 3).parameters())
    par_ada = sum(p.numel() for p in CodificadorAdaptativo(1, 1, 64, 3).parameters())
    print(f"[autoteste] parâmetros: residual {par_res:,} · adaptativo {par_ada:,} "
          f"({'iguais' if par_res == par_ada else 'DIFEREM - a comparação deixaria de isolar a máscara'})")


if __name__ == "__main__":
    _autoteste()


In [ ]:
%%writefile fase6_amplitude.py
import torch
import torch.nn as nn

import fase0_codificador_decodificador as base
import fase25_residual as f25
from fase6_adaptativo import textura_local

NIVEIS = None
QUANTIZAR = True
K_POR_BLOCO = None
BLOCO = 8


@torch.no_grad()
def mascara_esparsa(portadora, k=None, bloco=None):
    k = k or K_POR_BLOCO
    bloco = bloco or BLOCO
    t = textura_local(portadora)
    b, c, h, w = t.shape
    blocos = (t.view(b, c, h // bloco, bloco, w // bloco, bloco)
               .permute(0, 1, 2, 4, 3, 5).reshape(b, c, h // bloco, w // bloco, bloco * bloco))
    m = torch.zeros_like(blocos).scatter_(-1, blocos.topk(k, dim=-1).indices, 1.0)
    return (m.view(b, c, h // bloco, w // bloco, bloco, bloco)
             .permute(0, 1, 2, 4, 3, 5).reshape(b, c, h, w))


def arredondar_8bits(y):
    return y + (torch.round(y * 255.0) / 255.0 - y).detach()


class CodificadorAmplitude(nn.Module):

    def __init__(self, canais, profundidade_dados, largura=32, blocos=3):
        super().__init__()
        self.nucleo = f25.CodificadorResidual(canais, profundidade_dados, largura, blocos)

    def load_state_dict(self, estado, strict=True):
        if estado and not any(k.startswith("nucleo.") for k in estado):
            estado = {f"nucleo.{k}": v for k, v in estado.items()}
        return super().load_state_dict(estado, strict=strict)

    def forward(self, portadora, mensagem):
        x = self.nucleo.features(portadora)
        x = self.nucleo.entrada(torch.cat([x, mensagem], dim=1))
        x = self.nucleo.residuais(x)
        residuo = self.nucleo.para_residuo(x)
        if NIVEIS is not None:
            a = NIVEIS / 255.0
            residuo = a * torch.tanh(residuo / a)
        if K_POR_BLOCO is not None:
            residuo = residuo * mascara_esparsa(portadora)
        estego = (portadora + residuo).clamp(0.0, 1.0)
        return arredondar_8bits(estego) if QUANTIZAR else estego


def _fabrica_codificador(canais, profundidade, largura=32):
    return CodificadorAmplitude(canais, profundidade, largura, f25.BLOCOS_RESIDUAIS)


f25.registrar("amplitude", _fabrica_codificador)


def usar(arquitetura, niveis="manter", quantizar=None, k="manter", bloco=None):
    global NIVEIS, QUANTIZAR, K_POR_BLOCO, BLOCO
    if niveis != "manter":
        NIVEIS = None if niveis is None else float(niveis)
    if quantizar is not None:
        QUANTIZAR = bool(quantizar)
    if k != "manter":
        K_POR_BLOCO = None if k is None else int(k)
    if bloco is not None:
        BLOCO = int(bloco)
    if arquitetura != "amplitude":
        return f25.usar(arquitetura)
    f25.usar("amplitude")
    teto = "sem teto" if NIVEIS is None else f"{NIVEIS:g} níveis"
    esp = ("todos os pixels" if K_POR_BLOCO is None
           else f"{K_POR_BLOCO} de {BLOCO * BLOCO} por bloco ({K_POR_BLOCO / BLOCO ** 2:.1%})")
    print(f"[arquitetura] gerador = amplitude  |  teto = {teto}  quantizar = {QUANTIZAR}  "
          f"alteráveis = {esp}")
    return arquitetura


@torch.no_grad()
def medir_alteracao(portadora, estego):
    d = (estego - portadora) * 255.0
    fora = ((estego * 255.0 - torch.round(estego * 255.0)).abs() > 1e-3).float().mean()
    return {"taxa": (torch.round(d) != 0).float().mean().item(),
            "rms": d.pow(2).mean().sqrt().item(),
            "linf": d.abs().max().item(),
            "fora": fora.item()}


def _autoteste():
    torch.manual_seed(0)
    x = torch.randint(0, 256, (2, 1, 32, 32)).float() / 255.0
    m = torch.rand(2, 1, 32, 32)
    global NIVEIS, QUANTIZAR
    for niveis, quant in ((None, False), (None, True), (2, True), (1, True)):
        NIVEIS, QUANTIZAR = niveis, quant
        c = CodificadorAmplitude(1, 1, 16, 1)
        with torch.no_grad():
            c.nucleo.para_residuo.weight.mul_(1e-3)
            c.nucleo.para_residuo.bias.fill_(3 / 255)
        y = c(x, m)
        y.sum().backward()
        g = c.nucleo.para_residuo.weight.grad.abs().sum().item()
        print(f"[autoteste] teto={niveis}  quantizar={quant} | {medir_alteracao(x, y)} "
              f"| gradiente {'ok' if g > 0 else 'MORTO'}")
    NIVEIS, QUANTIZAR = None, True

    x = torch.rand(2, 1, 32, 32)
    m = mascara_esparsa(x, k=8, bloco=8)
    por_bloco = m.view(2, 1, 4, 8, 4, 8).sum(dim=(3, 5))
    print(f"[autoteste] máscara esparsa k=8/64: fração {m.mean():.4f} (esperado 0,125) | "
          f"por bloco {por_bloco.min():.0f}-{por_bloco.max():.0f} (esperado 8-8)")
    t = textura_local(x)
    escolhido, resto = t[m == 1].mean(), t[m == 0].mean()
    print(f"[autoteste] textura média escolhida {escolhido:.4f} > resto {resto:.4f}: "
          f"{'ok' if escolhido > resto else 'ERRADO'}")


if __name__ == "__main__":
    _autoteste()


In [ ]:
%%writefile fase4_pe.py
import csv
import glob
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import fase0_codificador_decodificador as base
import fase25_residual as f25
import fase3_esteganalisador as f3
import fase6_amplitude as fa

CONFIG = {
    "pasta_dados": None,
    "pasta_saida": "runs/phase4",
    "espelho": None,
    "tamanho_img": 128,
    "semente_split": 42,
    "fracao_val": 0.05,
    "n_val_detector": 1000,
    "semente_detector": 1234,
    "epocas": 20,
    "pares_por_lote": 32,
    "lr": {"srnet": 1e-3, "srnet_cur": 1e-3, "effnet": 5e-4},
    "lr_fino": 5e-4,
    "curriculo": {"srnet_cur": (1.0, 0.4, 0.2)},
    "epocas_curriculo": 5,
    "decaimento": 1e-4,
    "usar_amp": True,
    "num_trabalhadores": 2,
    "dispositivo": "cuda" if torch.cuda.is_available() else "cpu",
    "max_imagens": None,
}


def pe_auc(scores, rotulos):
    s = np.asarray(scores, dtype=np.float64)
    y = np.asarray(rotulos, dtype=np.int64)
    ordem = np.argsort(-s, kind="mergesort")
    s, y = s[ordem], y[ordem]
    ultimos = np.r_[np.diff(s) != 0, True]
    vp = np.r_[0, np.cumsum(y)[ultimos]]
    fp = np.r_[0, np.cumsum(1 - y)[ultimos]]
    tpr, fpr = vp / y.sum(), fp / (len(y) - y.sum())
    erro = (fpr + (1.0 - tpr)) / 2.0
    pe = float(np.minimum(erro, 1.0 - erro).min())
    trap = getattr(np, "trapezoid", None) or np.trapz
    auc = float(trap(tpr, fpr))
    return pe, max(auc, 1.0 - auc)


def quarteto(scores, rotulos):
    s, y = np.asarray(scores), np.asarray(rotulos)
    p = (s > 0).astype(np.int64)
    vp, fp, fn = ((p == 1) & (y == 1)).sum(), ((p == 1) & (y == 0)).sum(), ((p == 0) & (y == 1)).sum()
    prec = vp / (vp + fp) if vp + fp else float("nan")
    rec = vp / (vp + fn) if vp + fn else float("nan")
    f1 = 2 * prec * rec / (prec + rec) if prec == prec and rec == rec and prec + rec else float("nan")
    return {"acuracia": float((p == y).mean()), "precisao": float(prec),
            "recall": float(rec), "f1": float(f1)}


def dividir(cfg):
    caminhos = []
    for ext in base.EXTENSOES_IMG:
        caminhos += glob.glob(os.path.join(cfg["pasta_dados"], "**", "*" + ext), recursive=True)
        caminhos += glob.glob(os.path.join(cfg["pasta_dados"], "**", "*" + ext.upper()), recursive=True)
    caminhos = sorted(set(caminhos))
    random.Random(cfg["semente_split"]).shuffle(caminhos)
    if cfg.get("max_imagens"):
        caminhos = caminhos[:cfg["max_imagens"]]
    n_teste = max(1, int(len(caminhos) * cfg["fracao_val"]))
    n_val = min(cfg["n_val_detector"], (len(caminhos) - n_teste) // 5)
    idx = np.arange(len(caminhos))
    return caminhos, idx[n_teste + n_val:], idx[n_teste:n_teste + n_val], idx[:n_teste]


@torch.no_grad()
def carregar_capas(caminhos, cfg, log=print):
    cache = os.path.join(cfg["pasta_saida"], "corpus", f"capas_{len(caminhos)}.pt")
    if os.path.exists(cache):
        return torch.load(cache)
    ds = base.DatasetImagens(caminhos, 1, cfg["tamanho_img"], treino=False)
    t0 = time.time()
    capas = torch.stack([torch.round(ds[i] * 255.0).to(torch.uint8) for i in range(len(ds))])
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    torch.save(capas, cache)
    log(f"[corpus] {len(capas)} portadoras em {time.time() - t0:.0f}s -> {cache}")
    return capas


def _configurar_amplitude(gerador):
    fa.NIVEIS = gerador.get("teto")
    fa.QUANTIZAR = True
    fa.K_POR_BLOCO = gerador.get("k")
    fa.BLOCO = gerador.get("bloco", 8)


@torch.no_grad()
def gerar_estego(gerador, capas, idx_teste, cfg, log=print):
    cache = os.path.join(cfg["pasta_saida"], "corpus", f"estego_{gerador['nome']}_{len(capas)}.pt")
    if os.path.exists(cache):
        return torch.load(cache)

    dev = cfg["dispositivo"]
    t0 = time.time()
    if gerador["tipo"] == "lsbm":
        estego, acc = _lsbm(capas, gerador.get("bpp", 0.0625)), float("nan")
    elif gerador["tipo"] == "hill":
        estego, acc = _hill(capas, gerador.get("bpp", 0.0625), dev), float("nan")
    else:
        ck = torch.load(gerador["ckpt"], map_location="cpu", weights_only=False)
        c = dict(ck["cfg"]); c["dispositivo"] = dev
        args = (c["canais"], c["profundidade_dados"], c["largura"], f25.BLOCOS_RESIDUAIS)
        if gerador["tipo"] == "amplitude":
            _configurar_amplitude(gerador)
            cod = fa.CodificadorAmplitude(*args)
        else:
            cod = f25.CodificadorResidual(*args)
        cod.load_state_dict(ck.get("codificador", ck.get("encoder")))
        dec = f25.DecodificadorResidual(*args)
        dec.load_state_dict(ck.get("decodificador", ck.get("decoder")))
        cod, dec = cod.to(dev).eval(), dec.to(dev).eval()

        gerador_rng = torch.Generator(device="cpu").manual_seed(cfg["semente_detector"])
        teste = set(idx_teste.tolist())
        partes, certos, total = [], 0, 0
        for i in range(0, len(capas), 64):
            x = capas[i:i + 64].to(dev).float() / 255.0
            tm = base._tamanho_msg(c)
            msg = torch.randint(0, 2, (len(x), c["profundidade_dados"], tm, tm),
                                generator=gerador_rng).float().to(dev)
            y = torch.round(cod(x, base.espalhar_mensagem(msg, c)).clamp(0, 1) * 255.0)
            partes.append(y.to(torch.uint8).cpu())
            no_teste = [j for j in range(len(x)) if i + j in teste]
            if no_teste:
                logits = base.agregar_logits(dec(y[no_teste] / 255.0), c)
                certos += ((logits > 0).float() == msg[no_teste]).sum().item()
                total += msg[no_teste].numel()
        estego, acc = torch.cat(partes), certos / max(total, 1)

    x = capas[idx_teste].float(); y = estego[idx_teste].float()
    d = y - x
    info = {"acuracia_bits": acc,
            "psnr": float(10 * math.log10(255.0 ** 2 / d.pow(2).mean().clamp_min(1e-12).item())),
            "taxa_alteracao": float((d != 0).float().mean()),
            "rms_niveis": float(d.pow(2).mean().sqrt())}
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    torch.save((estego, info), cache)
    log(f"[corpus] {gerador['nome']}: {len(estego)} estegos em {time.time() - t0:.0f}s | "
        f"teste: acurácia de bits {acc:.4f}  PSNR {info['psnr']:.2f}  "
        f"taxa {info['taxa_alteracao']:.1%}  RMS {info['rms_niveis']:.2f} níveis")
    return estego, info


def _lsbm(capas, bpp, semente=7):
    g = torch.Generator().manual_seed(semente)
    n, _, h, w = capas.shape
    usados = int(round(bpp * h * w))
    x = capas.view(n, -1).to(torch.int16).clone()
    for i in range(n):
        pos = torch.randperm(h * w, generator=g)[:usados]
        muda = torch.rand(usados, generator=g) < 0.5
        sinal = torch.where(torch.rand(usados, generator=g) < 0.5, -1, 1).to(torch.int16)
        v = x[i, pos]
        sinal = torch.where(v == 0, 1, torch.where(v == 255, -1, sinal)).to(torch.int16)
        x[i, pos] = v + sinal * muda.to(torch.int16)
    return x.clamp(0, 255).to(torch.uint8).view_as(capas)


CUSTO_MOLHADO = 1e10


def _pad_simetrico(x, p):
    h, w = x.shape[-2:]
    iy = torch.cat([torch.arange(p - 1, -1, -1), torch.arange(h), torch.arange(h - 1, h - 1 - p, -1)])
    ix = torch.cat([torch.arange(p - 1, -1, -1), torch.arange(w), torch.arange(w - 1, w - 1 - p, -1)])
    return x[..., iy.to(x.device), :][..., ix.to(x.device)]


def custo_hill(x):
    x = x.double()
    kb = torch.tensor([[-1., 2., -1.], [2., -4., 2.], [-1., 2., -1.]], dtype=x.dtype,
                      device=x.device).view(1, 1, 3, 3)
    l1 = torch.full((1, 1, 3, 3), 1 / 9, dtype=x.dtype, device=x.device)
    l2 = torch.full((1, 1, 15, 15), 1 / 225, dtype=x.dtype, device=x.device)
    r = F.conv2d(_pad_simetrico(x, 1), kb).abs()
    rho = 1.0 / (F.conv2d(_pad_simetrico(r, 1), l1) + 1e-10)
    rho = F.conv2d(_pad_simetrico(rho, 7), l2).clamp_max(CUSTO_MOLHADO)
    return (torch.where(x >= 255, CUSTO_MOLHADO, rho),
            torch.where(x <= 0, CUSTO_MOLHADO, rho))


def _probabilidades(rp, rm, lam):
    ep, em = torch.exp(-lam * rp), torch.exp(-lam * rm)
    z = 1.0 + ep + em
    return ep / z, em / z


def _entropia_bits(pp, pm):
    p0 = 1.0 - pp - pm
    h = -(torch.special.xlogy(pp, pp) + torch.special.xlogy(pm, pm) + torch.special.xlogy(p0, p0))
    return h.flatten(1).sum(1) / math.log(2)


def simular_embutimento(x, rp, rm, bits, gerador_rng, iteracoes=60):
    n = x.shape[0]
    lo = torch.full((n, 1, 1, 1), -40.0, dtype=torch.float64, device=x.device)
    hi = torch.full((n, 1, 1, 1), 40.0, dtype=torch.float64, device=x.device)
    for _ in range(iteracoes):
        meio = (lo + hi) / 2
        h = _entropia_bits(*_probabilidades(rp, rm, meio.exp())).view(n, 1, 1, 1)
        acima = h > bits
        lo, hi = torch.where(acima, meio, lo), torch.where(acima, hi, meio)
    pp, pm = _probabilidades(rp, rm, ((lo + hi) / 2).exp())
    u = torch.rand(x.shape, generator=gerador_rng, dtype=torch.float64).to(x.device)
    delta = (u < pp).to(torch.int16) - ((u >= pp) & (u < pp + pm)).to(torch.int16)
    return delta


@torch.no_grad()
def _hill(capas, bpp, dispositivo, semente=7, lote=250):
    g = torch.Generator().manual_seed(semente)
    h, w = capas.shape[-2:]
    bits = bpp * h * w
    partes = []
    for i in range(0, len(capas), lote):
        x = capas[i:i + lote].to(dispositivo)
        rp, rm = custo_hill(x.double())
        delta = simular_embutimento(x, rp, rm, bits, g)
        partes.append((x.to(torch.int16) + delta).clamp(0, 255).to(torch.uint8).cpu())
    return torch.cat(partes)


class _EffNet(nn.Module):

    def __init__(self, pretreinada=True):
        super().__init__()
        import timm
        try:
            self.rede = timm.create_model("efficientnet_b0", pretrained=pretreinada,
                                          in_chans=1, num_classes=1)
            self.pretreinada = pretreinada
        except Exception as erro:
            print(f"[effnet] pesos do ImageNet indisponíveis ({type(erro).__name__}); do zero")
            self.rede = timm.create_model("efficientnet_b0", pretrained=False,
                                          in_chans=1, num_classes=1)
            self.pretreinada = False
        self.rede.conv_stem.stride = (1, 1)

    def forward(self, x):
        return self.rede((x - 0.449) / 0.226).squeeze(1)


def construir_detector(nome):
    if nome in ("srnet", "srnet_cur"):
        return f3.SRNet(1, compacta=False)
    if nome == "effnet":
        return _EffNet()
    raise ValueError(f"detector desconhecido: {nome}")


def _aumentar(x):
    k = random.randrange(4)
    if k:
        x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if random.random() < 0.5 else x


@torch.no_grad()
def _pontuar(rede, capas, estego, idx, cfg):
    dev, amp = cfg["dispositivo"], cfg["usar_amp"] and cfg["dispositivo"] == "cuda"
    rede.eval()
    scores, rotulos = [], []
    for i in range(0, len(idx), 128):
        b = idx[i:i + 128]
        x = torch.cat([capas[b], estego[b]]).to(dev).float() / 255.0
        with torch.autocast("cuda", enabled=amp):
            s = rede(x).float().cpu()
        scores.append(s)
        rotulos.append(torch.cat([torch.zeros(len(b)), torch.ones(len(b))]))
    return torch.cat(scores).numpy(), torch.cat(rotulos).numpy()


def treinar_detector(nome, capas, estego, idx_tr, idx_va, idx_te, cfg, log=print,
                     inicial=None, epocas=None, rotulo=None):
    torch.manual_seed(cfg["semente_detector"]); random.seed(cfg["semente_detector"])
    np.random.seed(cfg["semente_detector"])
    dev, amp = cfg["dispositivo"], cfg["usar_amp"] and cfg["dispositivo"] == "cuda"
    epocas = epocas or cfg["epocas"]
    rotulo = rotulo or nome
    rede = construir_detector(nome).to(dev)
    if inicial is not None:
        rede.load_state_dict(inicial)
    lr = cfg["lr_fino"] if inicial is not None else cfg["lr"][nome]
    opt = torch.optim.AdamW(rede.parameters(), lr=lr, weight_decay=cfg["decaimento"])
    pl = cfg["pares_por_lote"]
    passos = epocas * (len(idx_tr) // pl)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=passos, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    capas_dev, estego_dev = capas.to(dev), estego.to(dev)

    melhor = {"pe": 2.0}
    t_inicio = time.time()
    for epoca in range(1, epocas + 1):
        rede.train()
        perm = np.random.permutation(idx_tr)
        soma, n = 0.0, 0
        for i in range(0, len(perm) - pl + 1, pl):
            b = torch.as_tensor(perm[i:i + pl], device=dev)
            x = _aumentar(torch.cat([capas_dev[b], estego_dev[b]]).float() / 255.0)
            y = torch.cat([torch.zeros(pl, device=dev), torch.ones(pl, device=dev)])
            with torch.autocast("cuda", enabled=amp):
                perda = F.binary_cross_entropy_with_logits(rede(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(perda).backward(); scaler.step(opt); scaler.update(); sched.step()
            soma += perda.item(); n += 1

        pe_va, auc_va = pe_auc(*_pontuar(rede, capas, estego, idx_va, cfg))
        marca = ""
        if pe_va <= melhor["pe"]:
            melhor = {"pe": pe_va, "epoca": epoca,
                      "estado": {k: v.detach().cpu().clone() for k, v in rede.state_dict().items()}}
            marca = "  *"
        restam = (epocas - epoca) * (time.time() - t_inicio) / epoca
        log(f"[{rotulo:6s}] época {epoca:2d}  bce={soma / max(n, 1):.4f}  "
            f"val: P_E={pe_va:.4f}  AUC={auc_va:.4f}  restam ~{restam / 60:.0f} min{marca}")

    if idx_te is None:
        return {"pe_val": melhor["pe"], "epoca_escolhida": melhor["epoca"], "estado": melhor["estado"]}
    rede.load_state_dict(melhor["estado"])
    scores, rotulos = _pontuar(rede, capas, estego, idx_te, cfg)
    pe, auc = pe_auc(scores, rotulos)
    return {"pe": pe, "auc": auc, "epoca_escolhida": melhor["epoca"], **quarteto(scores, rotulos),
            "pretreinada": getattr(rede, "pretreinada", "n/a"), "estado": melhor["estado"]}


def _descrever_curriculo(nome, cfg):
    cargas = cfg["curriculo"].get(nome)
    if not cargas:
        return ""
    return ">".join(f"{c:g}" for c in cargas) + f" bpp x{cfg['epocas_curriculo']}ep"


def preparar_curriculo(nome, capas, idx_tr, idx_va, idx_te, cfg, log=print):
    cargas = cfg["curriculo"][nome]
    arquivo = (f"curriculo_{nome}_{'-'.join(f'{c:g}' for c in cargas)}_"
               f"{cfg['epocas_curriculo']}ep_{len(capas)}.pt")
    for pasta in (cfg["pasta_saida"], cfg["espelho"]):
        if pasta and os.path.exists(os.path.join(pasta, arquivo)):
            log(f"[curric] {nome}: pré-treino reaproveitado de {os.path.join(pasta, arquivo)}")
            return torch.load(os.path.join(pasta, arquivo))

    estado = None
    for carga in cargas:
        estego, _ = gerar_estego({"nome": f"lsbm{carga:g}", "tipo": "lsbm", "bpp": carga},
                                 capas, idx_te, cfg, log)
        log(f"[curric] {nome}: estágio LSBM {carga:g} bpp "
            f"({'do zero' if estado is None else 'a partir do estágio anterior'})")
        r = treinar_detector(nome, capas, estego, idx_tr, idx_va, None, cfg, log, inicial=estado,
                             epocas=cfg["epocas_curriculo"], rotulo=f"{nome}@{carga:g}")
        estado = r["estado"]
        aviso = ("  <- NÃO APRENDEU (P_E ~0,5): o estágio seguinte parte de pesos inúteis"
                 if r["pe_val"] > 0.45 else "")
        log(f"[curric] {nome}: LSBM {carga:g} bpp -> P_E de val {r['pe_val']:.4f} "
            f"(época {r['epoca_escolhida']}){aviso}")
    torch.save(estado, os.path.join(cfg["pasta_saida"], arquivo))
    return estado


COLUNAS = ["gerador", "detector", "pe", "auc", "acuracia", "precisao", "recall", "f1",
           "epoca_escolhida", "pretreinada", "curriculo", "acuracia_bits", "psnr",
           "taxa_alteracao", "rms_niveis", "epocas", "n_treino", "n_val", "n_teste"]


def _ler_tabela(caminho):
    if not os.path.exists(caminho):
        return []
    with open(caminho, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def rodar(geradores, detectores=("srnet", "effnet"), cfg=None):
    cfg = dict(CONFIG, **(cfg or {}))
    os.makedirs(cfg["pasta_saida"], exist_ok=True)
    arquivo_log = open(os.path.join(cfg["pasta_saida"], "fase4.log"), "a", encoding="utf-8")

    def log(msg):
        print(msg); arquivo_log.write(msg + "\n"); arquivo_log.flush()

    tabela_local = os.path.join(cfg["pasta_saida"], "resultados_fase4.csv")
    tabela_espelho = os.path.join(cfg["espelho"], "resultados_fase4.csv") if cfg["espelho"] else None
    feitas = _ler_tabela(tabela_espelho) if tabela_espelho else []
    feitas = feitas or _ler_tabela(tabela_local)
    ja = {(l["gerador"], l["detector"]) for l in feitas if int(l.get("epocas", 0)) == cfg["epocas"]}

    caminhos, idx_tr, idx_va, idx_te = dividir(cfg)
    log(f"[split ] {len(idx_tr)} treino_det / {len(idx_va)} val_det / {len(idx_te)} teste  "
        f"(teste = validação dos geradores, semente {cfg['semente_split']})")
    capas = carregar_capas(caminhos, cfg, log)

    iniciais = {}
    for det in detectores:
        falta = any((g["nome"], det) not in ja for g in geradores)
        if det in cfg["curriculo"] and falta:
            log("=" * 78)
            log(f"FASE 4 - curriculum do detector {det}: {_descrever_curriculo(det, cfg)}")
            log("=" * 78)
            iniciais[det] = preparar_curriculo(det, capas, idx_tr, idx_va, idx_te, cfg, log)
            f25.espelhar(cfg["pasta_saida"], cfg["espelho"])

    linhas = list(feitas)
    for gerador in geradores:
        pendentes = [d for d in detectores if (gerador["nome"], d) not in ja]
        if not pendentes:
            log(f"[pulado] {gerador['nome']}: todos os detectores já medidos")
            continue
        estego, info = gerar_estego(gerador, capas, idx_te, cfg, log)
        for det in pendentes:
            log("=" * 78)
            log(f"FASE 4 - gerador {gerador['nome']}  ×  detector {det}")
            log("=" * 78)
            r = treinar_detector(det, capas, estego, idx_tr, idx_va, idx_te, cfg, log,
                                 inicial=iniciais.get(det))
            pesos = os.path.join(cfg["pasta_saida"], f"detector_{gerador['nome']}_{det}.pt")
            torch.save(r.pop("estado"), pesos)
            linha = {"gerador": gerador["nome"], "detector": det, **r, **info,
                     "curriculo": _descrever_curriculo(det, cfg),
                     "epocas": cfg["epocas"], "n_treino": len(idx_tr), "n_val": len(idx_va),
                     "n_teste": len(idx_te)}
            linhas.append({k: linha.get(k, "") for k in COLUNAS})
            with open(tabela_local, "w", newline="", encoding="utf-8") as f:
                w = csv.DictWriter(f, fieldnames=COLUNAS); w.writeheader(); w.writerows(linhas)
            log(f"[teste ] {gerador['nome']} × {det}: P_E={r['pe']:.4f}  AUC={r['auc']:.4f}  "
                f"acurácia={r['acuracia']:.4f}  (época {r['epoca_escolhida']})")
            f25.espelhar(cfg["pasta_saida"], cfg["espelho"])
    arquivo_log.close()
    imprimir_tabela(linhas)
    return linhas


def imprimir_tabela(linhas):
    if not linhas:
        return
    dets = sorted({l["detector"] for l in linhas})
    gers = list(dict.fromkeys(l["gerador"] for l in linhas))
    val = {(l["gerador"], l["detector"]): l for l in linhas}
    fmt = lambda v, f: (f"{float(v):{f}}" if v not in ("", None) and float(v) == float(v) else "--")
    print("\n" + "=" * 78)
    print("FASE 4 - P_E no teste (0,5 = indetectável; 0 = detectado sempre)")
    print("=" * 78)
    print(f"{'gerador':<16}" + "".join(f"{'P_E ' + d:>14}" for d in dets)
          + f"{'acc bits':>10}{'PSNR':>8}{'taxa':>8}")
    for g in gers:
        info = next(val[(g, d)] for d in dets if (g, d) in val)
        print(f"{g:<16}" + "".join(f"{fmt(val[(g, d)]['pe'], '.4f') if (g, d) in val else '--':>14}"
                                   for d in dets)
              + f"{fmt(info['acuracia_bits'], '.4f'):>10}{fmt(info['psnr'], '.2f'):>8}"
              + f"{fmt(info['taxa_alteracao'], '.1%'):>8}")
    print("=" * 78)


## 4. Dataset
Baixa a BOSSBase; pula se já estiver no Drive.

In [ ]:
import os, subprocess

destino = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01')
if os.path.isdir(destino) and len(os.listdir(destino)) > 100:
    print('BOSSBase já está em', destino, '-', len(os.listdir(destino)), 'arquivos')
else:
    url = 'http://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip'
    print('Baixando BOSSBase (~2 min)...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', '/content/boss.zip', url], check=True)
    subprocess.run(['unzip', '-q', '/content/boss.zip', '-d', PASTA_DADOS_BASE], check=True)
    print('Pronto:', len(os.listdir(destino)), 'imagens')


### 4b. Cópia local
Copia a BOSSBase para o disco local antes dos treinos.

In [ ]:
import glob, os, shutil, time

ORIGEM_BOSS = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01')
LOCAL_BOSS  = '/content/datasets/BOSSbase_1.01'

n_origem = len(glob.glob(os.path.join(ORIGEM_BOSS, '*.pgm')))
if len(glob.glob(LOCAL_BOSS + '/*.pgm')) < n_origem:
    t0 = time.time()
    shutil.copytree(ORIGEM_BOSS, LOCAL_BOSS, dirs_exist_ok=True)
    print(f'copiado em {(time.time() - t0) / 60:.1f} min')

PASTA_DADOS_BASE = '/content/datasets'
PASTA_TRABALHO   = '/content/runs'
os.makedirs(PASTA_TRABALHO, exist_ok=True)
print(len(glob.glob(LOCAL_BOSS + '/*.pgm')), 'imagens em', LOCAL_BOSS)
print('dados   ->', PASTA_DADOS_BASE)
print('trabalho->', PASTA_TRABALHO, ' (espelhado em', PASTA_SAIDA_BASE + ')')

## 5. Checkpoint da Fase 2
Procura no Drive o `ckpt_best.pt` do gerador residual.

In [ ]:
import glob, os, torch

candidatos = sorted(glob.glob(os.path.join(BASE, '**', 'ckpt_best.pt'), recursive=True))
print('Checkpoints encontrados no Drive:')
for c in candidatos:
    try:
        e = torch.load(c, map_location='cpu')
        ep = e.get('epoca', e.get('epoch', '?'))
        tem = [k for k in ('codificador', 'decodificador', 'critico', 'srnet') if k in e]
        val = e.get('val', {})
        extra = ('psnr=%.2f' % val['psnr']) if 'psnr' in val else ''
        print('  %-70s ép.%-5s %s %s' % (c, ep, tem, extra))
    except Exception as ex:
        print('  %-70s [erro ao ler: %s]' % (c, ex))

CKPT_FASE2 = os.path.join(PASTA_SAIDA_BASE, 'phase2_residual', 'ckpt_best.pt')
print('\nUsando:', CKPT_FASE2, '->', 'OK' if os.path.exists(CKPT_FASE2) else 'NÃO EXISTE - corrija acima')

PSNR_PARTIDA = torch.load(CKPT_FASE2, map_location='cpu')['val']['psnr'] if os.path.exists(CKPT_FASE2) else 38.27
ALVO_PSNR = round(PSNR_PARTIDA - 1.0, 2)
print(f'PSNR de partida {PSNR_PARTIDA:.2f} dB -> critério (2): PSNR >= {ALVO_PSNR:.2f} dB')


## 6. Sanidade
3 épocas com a SRNet compacta, em pasta separada.

In [ ]:
import importlib, os
import fase0_codificador_decodificador, fase2_wgan_gp, fase3_esteganalisador as f3
for m in (fase0_codificador_decodificador, fase2_wgan_gp, f3):
    importlib.reload(m)

f3.CONFIG.update(f3.COLAB_GPU)
f3.CONFIG.update(
    pasta_dados     = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
    canais          = 1,
    carregar_fase2  = CKPT_FASE2,
    pasta_saida     = os.path.join(PASTA_TRABALHO, 'phase3_sanidade'),
    espelho_ckpt    = os.path.join(PASTA_SAIDA_BASE, 'phase3_sanidade'),
    epocas          = 3,
    srnet_compacta  = True,
    max_imagens     = 2000,
)
_ = f3.treinar()


## 6.4 Custo por época
Roda 2 épocas da Fase 2 com cada gerador (denso e residual) e mede s/época.

In [ ]:
import importlib, os, time
import fase0_codificador_decodificador as base, fase2_wgan_gp as fase2, fase25_residual as f25

EPOCAS_CUSTO = 2
custo = {}
for arq in ('denso', 'residual'):
    for m in (base, fase2, f25):
        importlib.reload(m)
    f25.usar(arq)
    t0 = time.time()
    f25.treinar_fase2(
        pasta_saida     = os.path.join(PASTA_TRABALHO, f'phase25_custo_{arq}'),
        espelho         = os.path.join(PASTA_SAIDA_BASE, f'phase25_custo_{arq}'),
        pasta_dados     = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
        canais          = 1,
        max_imagens     = None,
        epocas          = EPOCAS_CUSTO,
        carregar_fase0  = None,
        salvar_cada     = 99,
    )
    custo[arq] = (time.time() - t0) / EPOCAS_CUSTO

print('\n' + '=' * 70)
print(f'CUSTO POR ÉPOCA - Fase 2, dataset inteiro, mesma sessão ({torch.cuda.get_device_name(0)})')
for arq, s in custo.items():
    print(f'  {arq:<9} {s:7.1f} s/época')
print(f"  razão residual/denso: {custo['residual']/custo['denso']:.2f}x")
print(f"\nETA da sonda 6.5 (2 braços x 10 ép., SRNet ~+22%): ~{2*10*custo['residual']*1.22/3600:.1f} h")
print('Registre estes números no ROADMAP (Fase 2.5, critério 3).')

## 6.5 Sonda de `lambda_srnet`
Treina braços com λ = 0,05 e 0,1 e classifica o resultado em H1, H2 ou degenerado.

In [ ]:
import importlib, os, csv
import fase0_codificador_decodificador, fase2_wgan_gp, fase3_esteganalisador as f3
import fase25_residual as f25

BRACOS  = [0.05, 0.1]
EPOCAS  = 10
resumo  = {}

for lam in BRACOS:
    for m in (fase0_codificador_decodificador, fase2_wgan_gp, f3):
        importlib.reload(m)

    saida   = os.path.join(PASTA_TRABALHO, f'phase3_sonda_lam{lam:g}')
    espelho = os.path.join(PASTA_SAIDA_BASE, f'phase3_sonda_lam{lam:g}')
    f3.CONFIG.update(f3.COLAB_GPU)
    f3.CONFIG.update(
        pasta_dados     = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
        canais          = 1,
        carregar_fase2  = CKPT_FASE2,
        pasta_saida     = saida,
        espelho_ckpt    = espelho,
        max_imagens     = None,
        epocas          = EPOCAS,
        srnet_compacta  = False,
        alvo_srnet      = 'equilibrio',
        lambda_srnet    = lam,
    )
    print('\n' + '=' * 78)
    print(f'BRAÇO lambda_srnet = {lam}   ->   {saida}')
    print('=' * 78)
    resumo[lam] = {'melhor': f3.treinar(), 'saida': saida}
    f25.espelhar(saida, espelho)

print('\n' + '=' * 78)
print('SONDA - trajetória da AUC por época (o número que decide H1 vs H2)')
print('=' * 78)

BASE_PSNR = PSNR_PARTIDA

for lam in BRACOS:
    caminho = os.path.join(resumo[lam]['saida'], 'metricas_val.csv')
    if not os.path.exists(caminho):
        print(f'\nlambda={lam}: metricas_val.csv não encontrado em {caminho}')
        continue
    with open(caminho, encoding='utf-8') as f:
        linhas = list(csv.DictReader(f))
    aucs  = [max(float(l['auc_srnet']), 1 - float(l['auc_srnet'])) for l in linhas]
    psnrs = [float(l['psnr'])         for l in linhas]
    accs  = [float(l['acuracia_bits'])for l in linhas]

    print(f'\n--- lambda_srnet = {lam} ---')
    print('  ép    ' + ' '.join(f'{int(l["epoca"]):6d}' for l in linhas))
    print('  AUCef ' + ' '.join(f'{a:6.4f}' for a in aucs))
    print('  PSNR  ' + ' '.join(f'{p:6.2f}' for p in psnrs))
    print('  acc   ' + ' '.join(f'{a:6.4f}' for a in accs))

    psnr_fim  = sum(psnrs[-3:]) / len(psnrs[-3:])
    auc_min   = min(aucs)
    auc_fim   = sum(aucs[-3:]) / len(aucs[-3:])
    acc_fim   = sum(accs[-3:]) / len(accs[-3:])
    valido    = acc_fim >= 0.99 and psnr_fim >= ALVO_PSNR
    saiu      = auc_fim < 0.9990
    queda     = BASE_PSNR - psnr_fim
    resumo[lam].update(auc_min=auc_min, auc_fim=auc_fim, saiu=saiu, valido=valido, acc_fim=acc_fim, psnr_fim=psnr_fim, queda=queda,
                       acc_min=min(accs))
    print(f'  -> AUC média das 3 últimas {auc_fim:.4f} (mínima {auc_min:.4f}) | PSNR médio das 3 últimas '
          f'{psnr_fim:.2f} dB (queda de {queda:+.2f} dB vs {BASE_PSNR:.2f}) | acc média das 3 últimas {acc_fim:.4f}'
          + ('' if valido else '  <-- MENSAGEM PERDIDA ou PSNR abaixo do alvo: braço sem valor'))

print('\n' + '=' * 78)
moveu = [l for l in BRACOS if resumo.get(l, {}).get('saiu') and resumo[l]['valido']]
degenerados = [l for l in BRACOS if resumo.get(l, {}).get('saiu') and not resumo[l]['valido']]
if moveu:
    melhor_lam = min(moveu, key=lambda l: resumo[l]['auc_fim'])
    print(f'H1 SUSTENTADA: a AUC saiu de 1,0 com lambda_srnet={melhor_lam} '
          f'(média das 3 últimas {resumo[melhor_lam]["auc_fim"]:.4f}).')
    print(f'Próximo passo: 60 épocas com lambda_srnet={melhor_lam}, vigiando o critério (2) -')
    print(f'o PSNR neste braço já caiu {resumo[melhor_lam]["queda"]:+.2f} dB em apenas {EPOCAS} épocas.')
    print(f'Se o PSNR furar {ALVO_PSNR:.2f} antes de a AUC chegar perto de 0,5, suba lambda_imagem junto.')
elif degenerados:
    print('NENHUM BRAÇO VÁLIDO: a AUC só caiu onde a mensagem se perdeu '
          f'(lambda {degenerados}; acc média das 3 últimas: '
          + ', '.join(f"{resumo[l]['acc_fim']:.3f}" for l in degenerados) + ').')
    print('O gerador enganou o detector deixando de esconder - não é H1.')
    print('Leitura: nenhum peso testado preserva a mensagem E tira a AUC de 1,0 -> inclina para H2.')
    print('Próximos passos baratos: (a) sonda de payload tamanho_msg=16 (0,0156 bpp) com o')
    print('lambda que preserva a mensagem; (b) um braço de lambda intermediário (0,05-0,1) para')
    print('confirmar que não existe ponto útil entre 0,01 e 0,5.')
else:
    piores = ', '.join(f'lambda={l}: PSNR {resumo[l]["psnr_fim"]:.2f} dB '
                       f'({resumo[l]["queda"]:+.2f})' for l in BRACOS if l in resumo)
    print('H2 SUSTENTADA: a AUC terminou cravada em ~1,0 nos dois braços.')
    print(f'  {piores}')
    print('Pare de tunar lambda_srnet - o peso não é o gargalo. Duas saídas, nesta ordem:')
    print('  (a) sonda de payload: tamanho_msg=16 (0,0156 bpp), 10 épocas. Se a AUC continuar')
    print('      em 1,0 mesmo a 1/4 do payload, o problema é a ARQUITETURA, não a carga.')
    print('  (b) Fase 6 (carga adaptativa por textura): decidir ONDE os bits vão, em vez de')
    print('      espalhá-los uniformemente. É a limitação que o artigo do SBSeg já registrava.')
print('Em qualquer dos casos, a Fase 4 (P_E independente) segue sendo o instrumento de medida.')
print('=' * 78)


## 6.6 Sonda de carga
Repete o braço λ = 0,05 com `tamanho_msg = 16` (0,0156 bpp).

In [ ]:
import importlib, os, csv
import fase0_codificador_decodificador, fase2_wgan_gp, fase3_esteganalisador as f3
import fase25_residual as f25

MSG_SONDA    = 16
LAMBDA_SONDA = 0.05
EPOCAS       = 10

for m in (fase0_codificador_decodificador, fase2_wgan_gp, f3):
    importlib.reload(m)

saida   = os.path.join(PASTA_TRABALHO, f'phase3_carga_msg{MSG_SONDA}')
espelho = os.path.join(PASTA_SAIDA_BASE, f'phase3_carga_msg{MSG_SONDA}')
f3.CONFIG.update(f3.COLAB_GPU)
f3.CONFIG.update(
    pasta_dados     = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
    canais          = 1,
    carregar_fase2  = CKPT_FASE2,
    pasta_saida     = saida,
    espelho_ckpt    = espelho,
    max_imagens     = None,
    epocas          = EPOCAS,
    srnet_compacta  = False,
    alvo_srnet      = 'equilibrio',
    lambda_srnet    = LAMBDA_SONDA,
    tamanho_msg     = MSG_SONDA,
)
print('=' * 78)
print(f'SONDA DE CARGA - tamanho_msg={MSG_SONDA} ({(MSG_SONDA/128)**2:.4g} bpp), '
      f'lambda_srnet={LAMBDA_SONDA}   ->   {saida}')
print('=' * 78)
resultado_carga = f3.treinar()
f25.espelhar(saida, espelho)


def ler_curva(pasta):
    caminho = os.path.join(pasta, 'metricas_val.csv')
    if not os.path.exists(caminho):
        return None
    with open(caminho, encoding='utf-8') as f:
        linhas = list(csv.DictReader(f))
    return {
        'auc':  [max(float(l['auc_srnet']), 1 - float(l['auc_srnet'])) for l in linhas],
        'psnr': [float(l['psnr']) for l in linhas],
        'acc':  [float(l['acuracia_bits']) for l in linhas],
    }

def media3(v):
    return sum(v[-3:]) / len(v[-3:])

ref = None
for p in (os.path.join(PASTA_TRABALHO, f'phase3_sonda_lam{LAMBDA_SONDA:g}'),
          os.path.join(PASTA_SAIDA_BASE, f'phase3_sonda_lam{LAMBDA_SONDA:g}')):
    ref = ref or ler_curva(p)
sonda = ler_curva(saida)

print('\n' + '=' * 78)
print('SONDA DE CARGA - quantos bits (carga) x onde os bits vão (distribuição)')
print('=' * 78)
print(f"{'':<26}{'carga cheia (32)':>18}{'sonda (' + str(MSG_SONDA) + ')':>18}")
print(f"{'bpp':<26}{0.0625:>18.4g}{(MSG_SONDA/128)**2:>18.4g}")
for rotulo, chave, fmt in (('AUC efetiva (3 últimas)', 'auc', '18.4f'),
                           ('acurácia (3 últimas)', 'acc', '18.4f'),
                           ('PSNR (3 últimas, dB)', 'psnr', '18.2f')):
    a = f'{media3(ref[chave]):{fmt}}' if ref else f"{'-- ':>18}"
    print(f'{rotulo:<26}{a}{media3(sonda[chave]):{fmt}}')

n_ep = len(sonda['auc'])
print('\n  ép    ' + ' '.join(f'{i+1:6d}' for i in range(n_ep)))
print('  AUCef ' + ' '.join(f'{a:6.4f}' for a in sonda['auc']))
print('  PSNR  ' + ' '.join(f'{p:6.2f}' for p in sonda['psnr']))
print('  acc   ' + ' '.join(f'{a:6.4f}' for a in sonda['acc']))

auc_fim, acc_fim = media3(sonda['auc']), media3(sonda['acc'])
print('\n' + '=' * 78)
if acc_fim < 0.99:
    print(f'BRAÇO SEM VALOR: acurácia média das 3 últimas = {acc_fim:.4f} < 0,99.')
    print('A mensagem se perdeu, então a AUC não diz nada. Baixe lambda_srnet e repita.')
elif auc_fim < 0.9990:
    print(f'A CARGA ERA O GARGALO: AUC efetiva caiu para {auc_fim:.4f} com a mensagem intacta '
          f'({acc_fim:.4f}).')
    print('Próximos passos: (a) sondar 8x8 (0,0039 bpp) para achar onde a AUC chega perto de 0,5;')
    print('(b) a curva segurança x capacidade da Fase 5 passa a ser o resultado central.')
else:
    print(f'NEM A 1/4 DA CARGA: AUC efetiva {auc_fim:.4f} com a mensagem intacta ({acc_fim:.4f}).')
    print('O gargalo não é QUANTOS bits, é ONDE eles vão - o resíduo espalhado uniformemente é')
    print('detectável em qualquer carga testada. A Fase 6 (carga adaptativa por textura) passa a')
    print('ser a resposta principal do trabalho, e não um incremento.')
print('A medida definitiva continua sendo a Fase 4 (P_E com detector independente).')
print('=' * 78)

## 6.7 Carga adaptativa (modo `global`)
Multiplica o resíduo por uma máscara de textura com média 1.

In [ ]:
import importlib, os, csv
import fase0_codificador_decodificador, fase2_wgan_gp, fase25_residual
import fase3_esteganalisador as f3
import fase6_adaptativo as f6

NORMALIZACAO = 'global'
EPOCAS       = 10
LAMBDA       = 0.05

for m in (fase0_codificador_decodificador, fase2_wgan_gp, fase25_residual, f3, f6):
    importlib.reload(m)
f6.usar('adaptativo', bloco=4, normalizacao=NORMALIZACAO)

saida   = os.path.join(PASTA_TRABALHO, f'phase6_adaptativo_{NORMALIZACAO}')
espelho = os.path.join(PASTA_SAIDA_BASE, f'phase6_adaptativo_{NORMALIZACAO}')
f3.CONFIG.update(f3.COLAB_GPU)
f3.CONFIG.update(
    pasta_dados     = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
    canais          = 1,
    carregar_fase2  = CKPT_FASE2,
    pasta_saida     = saida,
    espelho_ckpt    = espelho,
    max_imagens     = None,
    epocas          = EPOCAS,
    srnet_compacta  = False,
    alvo_srnet      = 'equilibrio',
    lambda_srnet    = LAMBDA,
    tamanho_msg     = 32,
    arquitetura     = 'adaptativo',
    retomar_de      = None,
)
print('=' * 78)
print(f'FASE 6 (triagem) - máscara de textura, normalizacao={NORMALIZACAO}  ->  {saida}')
print('=' * 78)
resultado_f6 = f3.treinar()
f6.usar('residual')
fase25_residual.espelhar(saida, espelho)


def curva(pasta):
    caminho = os.path.join(pasta, 'metricas_val.csv')
    if not os.path.exists(caminho):
        return None
    with open(caminho, encoding='utf-8') as f:
        linhas = list(csv.DictReader(f))
    return {'auc':  [max(float(l['auc_srnet']), 1 - float(l['auc_srnet'])) for l in linhas],
            'psnr': [float(l['psnr']) for l in linhas],
            'acc':  [float(l['acuracia_bits']) for l in linhas]}

media3 = lambda v: sum(v[-3:]) / len(v[-3:])
ref = None
for p in (os.path.join(PASTA_TRABALHO, f'phase3_sonda_lam{LAMBDA:g}'),
          os.path.join(PASTA_SAIDA_BASE, f'phase3_sonda_lam{LAMBDA:g}')):
    ref = ref or curva(p)
ada = curva(saida)

print('\n' + '=' * 78)
print('FASE 6 (triagem) - resíduo uniforme x concentrado na textura')
print('=' * 78)
print(f"{'':<28}{'uniforme':>14}{'adaptativo':>14}")
for rotulo, chave, fmt in (('AUC efetiva (3 últimas)', 'auc', '14.4f'),
                           ('acurácia (3 últimas)', 'acc', '14.4f'),
                           ('PSNR (3 últimas, dB)', 'psnr', '14.2f')):
    a = f'{media3(ref[chave]):{fmt}}' if ref else f"{'--':>14}"
    print(f'{rotulo:<28}{a}{media3(ada[chave]):{fmt}}')

n = len(ada['auc'])
print('\n  ép    ' + ' '.join(f'{i+1:6d}' for i in range(n)))
print('  AUCef ' + ' '.join(f'{a:6.4f}' for a in ada['auc']))
print('  PSNR  ' + ' '.join(f'{p:6.2f}' for p in ada['psnr']))
print('  acc   ' + ' '.join(f'{a:6.4f}' for a in ada['acc']))

auc_fim, acc_fim = media3(ada['auc']), media3(ada['acc'])
print('\n' + '=' * 78)
if acc_fim < 0.99:
    print(f'A MÁSCARA QUEBROU A LEITURA: acurácia {acc_fim:.4f} < 0,99.')
    print('Rode de novo com NORMALIZACAO = "bloco" (cada bit mantém seu orçamento) antes')
    print('de concluir qualquer coisa sobre indetectabilidade.')
elif auc_fim < 0.9990:
    print(f'A DISTRIBUIÇÃO IMPORTA: AUC efetiva {auc_fim:.4f} com a mensagem intacta '
          f'({acc_fim:.4f}).')
    print('A Fase 6 completa (mapa de custo APRENDIDO) passa a valer o investimento -')
    print('a máscara fixa já move a agulha, e um custo aprendido tende a mover mais.')
else:
    print(f'NEM CONCENTRANDO NA TEXTURA: AUC efetiva {auc_fim:.4f} com acurácia {acc_fim:.4f}.')
    print('Distribuir melhor não basta: o gargalo é a AMPLITUDE do resíduo (3,1 níveis de')
    print('cinza contra 0,18 do LSB clássico na mesma carga). O caminho passa a ser reduzir')
    print('a energia da perturbação - orçamento de distorção - ou trocar o paradigma de')
    print('embutimento (±1 nível com custo aprendido), e não refinar a máscara.')
print('A medida definitiva continua sendo a Fase 4 (P_E com detector independente).')
print('=' * 78)

## 6.8 Carga adaptativa (modo `bloco`)
Mesma máscara, normalizada com média 1 dentro de cada bloco.

In [ ]:
import importlib, os, csv, glob
import torch
import fase0_codificador_decodificador, fase2_wgan_gp, fase25_residual
import fase3_esteganalisador as f3
import fase6_adaptativo as f6

NORMALIZACAO = 'bloco'
EPOCAS       = 10
LAMBDA       = 0.05

for m in (fase0_codificador_decodificador, fase2_wgan_gp, fase25_residual, f3, f6):
    importlib.reload(m)

pasta_boss = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01')
caminhos = sorted(p for p in glob.glob(os.path.join(pasta_boss, '**', '*'), recursive=True)
                  if p.lower().endswith(fase0_codificador_decodificador.EXTENSOES_IMG))[-256:]
ds = fase0_codificador_decodificador.DatasetImagens(caminhos, 1, 128, treino=False)
x = torch.stack([ds[i] for i in range(len(ds))])
print(f'[máscara] {len(ds)} imagens de validação, bloco=4, janela={f6.JANELA}, '
      f'piso={f6.PISO}, teto={f6.TETO}')
for modo in ('global', 'bloco'):
    mk = f6.mascara_textura(x, bloco=4, modo=modo)
    e2 = (mk * mk).mean().item()
    print(f'[máscara] {modo:6s}  E[m]={mk.mean().item():.3f}  E[m²]={e2:.3f}  '
          f'-> penalidade de PSNR ~{10 * torch.log10(torch.tensor(e2)).item():.2f} dB  '
          f'| fração no piso {(mk <= f6.PISO * 1.01).float().mean().item():.1%}')
del x, ds

f6.usar('adaptativo', bloco=4, normalizacao=NORMALIZACAO)

saida   = os.path.join(PASTA_TRABALHO, f'phase6_adaptativo_{NORMALIZACAO}')
espelho = os.path.join(PASTA_SAIDA_BASE, f'phase6_adaptativo_{NORMALIZACAO}')
f3.CONFIG.update(f3.COLAB_GPU)
f3.CONFIG.update(
    pasta_dados     = pasta_boss,
    canais          = 1,
    carregar_fase2  = CKPT_FASE2,
    pasta_saida     = saida,
    espelho_ckpt    = espelho,
    max_imagens     = None,
    epocas          = EPOCAS,
    srnet_compacta  = False,
    alvo_srnet      = 'equilibrio',
    lambda_srnet    = LAMBDA,
    tamanho_msg     = 32,
    arquitetura     = 'adaptativo',
    retomar_de      = None,
)
print('=' * 78)
print(f'FASE 6 (controle) - máscara de textura, normalizacao={NORMALIZACAO}  ->  {saida}')
print('=' * 78)
resultado_f6_bloco = f3.treinar()
f6.usar('residual')
fase25_residual.espelhar(saida, espelho)


def curva(*pastas):
    for pasta in pastas:
        caminho = os.path.join(pasta, 'metricas_val.csv')
        if os.path.exists(caminho):
            with open(caminho, encoding='utf-8') as f:
                linhas = list(csv.DictReader(f))
            return {'auc':  [max(float(l['auc_srnet']), 1 - float(l['auc_srnet'])) for l in linhas],
                    'psnr': [float(l['psnr']) for l in linhas],
                    'acc':  [float(l['acuracia_bits']) for l in linhas]}
    return None

def ambos(nome):
    return (os.path.join(PASTA_TRABALHO, nome), os.path.join(PASTA_SAIDA_BASE, nome))

media3 = lambda v: sum(v[-3:]) / len(v[-3:])
bracos = {'uniforme': curva(*ambos(f'phase3_sonda_lam{LAMBDA:g}')),
          'global':   curva(*ambos('phase6_adaptativo_global')),
          'bloco':    curva(saida)}

print('\n' + '=' * 78)
print('FASE 6 (triagem) - uniforme x adaptativo global x adaptativo por bloco')
print('=' * 78)
print(f"{'':<28}" + ''.join(f'{k:>14}' for k in bracos))
for rotulo, chave, fmt in (('AUC efetiva (3 últimas)', 'auc', '14.4f'),
                           ('acurácia (3 últimas)', 'acc', '14.4f'),
                           ('PSNR (3 últimas, dB)', 'psnr', '14.2f')):
    print(f'{rotulo:<28}' + ''.join(f'{media3(c[chave]):{fmt}}' if c else f"{'--':>14}"
                                    for c in bracos.values()))

blo = bracos['bloco']
n = len(blo['auc'])
print('\n  bloco, por época')
print('  ép    ' + ' '.join(f'{i+1:6d}' for i in range(n)))
print('  AUCef ' + ' '.join(f'{a:6.4f}' for a in blo['auc']))
print('  PSNR  ' + ' '.join(f'{p:6.2f}' for p in blo['psnr']))
print('  acc   ' + ' '.join(f'{a:6.4f}' for a in blo['acc']))

auc_fim, acc_fim = media3(blo['auc']), media3(blo['acc'])
print('\n' + '=' * 78)
if acc_fim < 0.99:
    print(f'NEM O BLOCO SEGURA A LEITURA: acurácia {acc_fim:.4f} < 0,99.')
    print('Com a máscara quase plana isso não deveria acontecer - o problema não é a')
    print('redistribuição. Conferir o warm start e comparar com o braço uniforme.')
elif auc_fim < 0.9990:
    print(f'O BLOCO MOVEU A AUC: {auc_fim:.4f} com a mensagem intacta ({acc_fim:.4f}).')
    print('Inesperado, porque a máscara por bloco quase não redistribui. Antes de celebrar,')
    print('verificar se não é só menos energia (PSNR acima do uniforme).')
else:
    print(f'CONTROLE COMO ESPERADO: leitura intacta ({acc_fim:.4f}), AUC {auc_fim:.4f}.')
    print('O bloco confirma que a queda de acurácia do global veio de esvaziar os bits das')
    print('regiões lisas. A hipótese continua sem teste justo: o global precisa rodar com a')
    print('MESMA ENERGIA do uniforme (normalizar E[m²]=1, não E[m]=1) e piso menor.')
print('A medida definitiva continua sendo a Fase 4 (P_E com detector independente).')
print('=' * 78)

## 6.9 Carga adaptativa (energia)
Máscara global normalizada por E[m²] = 1.

In [ ]:
import importlib, os, csv
import fase0_codificador_decodificador, fase2_wgan_gp, fase25_residual
import fase3_esteganalisador as f3
import fase6_adaptativo as f6

NORMALIZACAO = 'global'
PRESERVAR    = 'energia'
EPOCAS       = 10
LAMBDA       = 0.05

for m in (fase0_codificador_decodificador, fase2_wgan_gp, fase25_residual, f3, f6):
    importlib.reload(m)
f6.usar('adaptativo', bloco=4, normalizacao=NORMALIZACAO, preservar=PRESERVAR)

nome    = f'phase6_adaptativo_{NORMALIZACAO}_{PRESERVAR}'
saida   = os.path.join(PASTA_TRABALHO, nome)
espelho = os.path.join(PASTA_SAIDA_BASE, nome)
f3.CONFIG.update(f3.COLAB_GPU)
f3.CONFIG.update(
    pasta_dados     = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
    canais          = 1,
    carregar_fase2  = CKPT_FASE2,
    pasta_saida     = saida,
    espelho_ckpt    = espelho,
    max_imagens     = None,
    epocas          = EPOCAS,
    srnet_compacta  = False,
    alvo_srnet      = 'equilibrio',
    lambda_srnet    = LAMBDA,
    tamanho_msg     = 32,
    arquitetura     = 'adaptativo',
    retomar_de      = None,
)
print('=' * 78)
print(f'FASE 6 (triagem) - global com E[m²]=1  ->  {saida}')
print('=' * 78)
resultado_f6_energia = f3.treinar()
f6.usar('residual')
fase25_residual.espelhar(saida, espelho)


def curva(*pastas):
    for pasta in pastas:
        caminho = os.path.join(pasta, 'metricas_val.csv')
        if os.path.exists(caminho):
            with open(caminho, encoding='utf-8') as f:
                linhas = list(csv.DictReader(f))
            return {'auc':  [max(float(l['auc_srnet']), 1 - float(l['auc_srnet'])) for l in linhas],
                    'psnr': [float(l['psnr']) for l in linhas],
                    'acc':  [float(l['acuracia_bits']) for l in linhas]}
    return None

def ambos(nome):
    return (os.path.join(PASTA_TRABALHO, nome), os.path.join(PASTA_SAIDA_BASE, nome))

media3 = lambda v: sum(v[-3:]) / len(v[-3:])
bracos = {'uniforme':     curva(*ambos(f'phase3_sonda_lam{LAMBDA:g}')),
          'glob/média':   curva(*ambos('phase6_adaptativo_global')),
          'bloco':        curva(*ambos('phase6_adaptativo_bloco')),
          'glob/energia': curva(saida)}

print('\n' + '=' * 78)
print('FASE 6 (triagem) - os quatro braços, média das 3 últimas épocas')
print('=' * 78)
print(f"{'':<16}" + ''.join(f'{k:>14}' for k in bracos))
for rotulo, chave, fmt in (('AUC efetiva', 'auc', '14.4f'),
                           ('acurácia', 'acc', '14.4f'),
                           ('PSNR (dB)', 'psnr', '14.2f')):
    print(f'{rotulo:<16}' + ''.join(f'{media3(c[chave]):{fmt}}' if c else f"{'--':>14}"
                                    for c in bracos.values()))

ene = bracos['glob/energia']
n = len(ene['auc'])
print('\n  global/energia, por época')
print('  ép    ' + ' '.join(f'{i+1:6d}' for i in range(n)))
print('  AUCef ' + ' '.join(f'{a:6.4f}' for a in ene['auc']))
print('  PSNR  ' + ' '.join(f'{p:6.2f}' for p in ene['psnr']))
print('  acc   ' + ' '.join(f'{a:6.4f}' for a in ene['acc']))

auc_fim, acc_fim, psnr_fim = media3(ene['auc']), media3(ene['acc']), media3(ene['psnr'])
uni = bracos['uniforme']
print('\n' + '=' * 78)
if uni and psnr_fim < media3(uni['psnr']) - 1.0:
    print(f'AVISO: PSNR {psnr_fim:.2f} dB, mais de 1 dB abaixo do uniforme '
          f'({media3(uni["psnr"]):.2f}). As energias voltaram a divergir no treino;')
    print('leia a AUC abaixo com essa ressalva.')
if auc_fim < 0.9990 and acc_fim >= 0.99:
    print(f'A DISTRIBUIÇÃO IMPORTA: AUC efetiva {auc_fim:.4f} com a mensagem intacta ({acc_fim:.4f}),')
    print('com a mesma energia do uniforme. O mapa de custo APRENDIDO (degrau 2) passa a')
    print('valer o investimento. Próximo: baixar o piso para ver quanto mais a AUC cede.')
elif auc_fim < 0.9990:
    print(f'A AUC CEDEU ({auc_fim:.4f}), MAS A LEITURA QUEBROU ({acc_fim:.4f} < 0,99).')
    print('A hipótese ganha força: concentrar na textura ajuda com a energia igual. O que')
    print('falta é dar orçamento aos bits das regiões lisas (ECC, lambda_mensagem, ou')
    print('embaralhar os bits para que nenhum dependa só de um bloco liso).')
else:
    print(f'NEM COM A MESMA ENERGIA: AUC efetiva {auc_fim:.4f}, acurácia {acc_fim:.4f}.')
    print('Distribuir o resíduo não move o detector: o gargalo é a AMPLITUDE (≈3,1 níveis')
    print('de cinza contra 0,18 do LSB na mesma carga). Próximo passo: orçamento explícito')
    print('de distorção ou embutimento ±1 com custo aprendido, e não refinar a máscara.')
print('A medida definitiva continua sendo a Fase 4 (P_E com detector independente).')
print('=' * 78)

## 6.10 Sonda de amplitude
Braços `quant`, `quant_t2` e `quant_t2_l1`: quantização em 8 bits, teto de amplitude e L1 do resíduo.

In [ ]:
import importlib, os, csv, glob
import torch
import fase0_codificador_decodificador, fase2_wgan_gp, fase25_residual
import fase3_esteganalisador as f3
import fase6_amplitude as fa

EPOCAS = 10
LAMBDA = 0.05
BRACOS = [
    ('quant',       None, 0.05),
    ('quant_t2',    2,    0.05),
    ('quant_t2_l1', 2,    1.0),
]
pasta_boss = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01')


def medir(saida):
    ck = next((os.path.join(saida, n) for n in ('ckpt_last.pt', 'ckpt_best.pt')
               if os.path.exists(os.path.join(saida, n))), None)
    if ck is None:
        return None
    estado = torch.load(ck, map_location='cpu', weights_only=False)
    cfg = dict(estado['cfg'])
    cod = fa.CodificadorAmplitude(cfg['canais'], cfg['profundidade_dados'], cfg['largura'],
                                  fase25_residual.BLOCOS_RESIDUAIS).to(cfg['dispositivo'])
    cod.load_state_dict(estado['codificador']); cod.eval()
    caminhos = sorted(p for p in glob.glob(os.path.join(pasta_boss, '**', '*'), recursive=True)
                      if p.lower().endswith(fase0_codificador_decodificador.EXTENSOES_IMG))[-256:]
    ds = fase0_codificador_decodificador.DatasetImagens(caminhos, 1, cfg['tamanho_img'], treino=False)
    x = torch.stack([ds[i] for i in range(len(ds))]).to(cfg['dispositivo'])
    torch.manual_seed(0)
    with torch.no_grad():
        msg = fase0_codificador_decodificador.mensagem_aleatoria(len(x), cfg)
        y = cod(x, fase0_codificador_decodificador.espalhar_mensagem(msg, cfg))
    return {**fa.medir_alteracao(x, y), 'epoca': estado.get('epoca')}


alteracao = {}
for nome, teto, lam_res in BRACOS:
    saida   = os.path.join(PASTA_TRABALHO, f'phase6_amp_{nome}')
    espelho = os.path.join(PASTA_SAIDA_BASE, f'phase6_amp_{nome}')
    for m in (fase0_codificador_decodificador, fase2_wgan_gp, fase25_residual, f3, fa):
        importlib.reload(m)
    fa.usar('amplitude', niveis=teto, quantizar=True)

    feito = os.path.join(espelho, 'metricas_val.csv')
    if os.path.exists(feito) and sum(1 for _ in open(feito, encoding='utf-8')) > EPOCAS:
        print(f'[pulado] {nome}: {EPOCAS} épocas já registradas em {espelho}')
    else:
        f3.CONFIG.update(f3.COLAB_GPU)
        f3.CONFIG.update(
            pasta_dados     = pasta_boss,
            canais          = 1,
            carregar_fase2  = CKPT_FASE2,
            pasta_saida     = saida,
            espelho_ckpt    = espelho,
            max_imagens     = None,
            epocas          = EPOCAS,
            srnet_compacta  = False,
            alvo_srnet      = 'equilibrio',
            lambda_srnet    = LAMBDA,
            lambda_residuo  = lam_res,
            tamanho_msg     = 32,
            arquitetura     = 'amplitude',
            retomar_de      = None,
        )
        print('=' * 78)
        print(f'SONDA DE AMPLITUDE - braço {nome}: teto={teto}  lambda_residuo={lam_res}  ->  {saida}')
        print('=' * 78)
        f3.treinar()
        fase25_residual.espelhar(saida, espelho)

    alteracao[nome] = medir(saida) or medir(espelho)
    fa.usar('residual')


def curva(*pastas):
    for pasta in pastas:
        caminho = os.path.join(pasta, 'metricas_val.csv')
        if os.path.exists(caminho):
            with open(caminho, encoding='utf-8') as f:
                linhas = list(csv.DictReader(f))
            return {'auc':  [max(float(l['auc_srnet']), 1 - float(l['auc_srnet'])) for l in linhas],
                    'psnr': [float(l['psnr']) for l in linhas],
                    'acc':  [float(l['acuracia_bits']) for l in linhas]}
    return None

def ambos(nome):
    return (os.path.join(PASTA_TRABALHO, nome), os.path.join(PASTA_SAIDA_BASE, nome))

media3 = lambda v: sum(v[-3:]) / len(v[-3:])
res = {'uniforme': curva(*ambos(f'phase3_sonda_lam{LAMBDA:g}'))}
res.update({nome: curva(*ambos(f'phase6_amp_{nome}')) for nome, _, _ in BRACOS})
alt = {'uniforme': {'taxa': 0.793, 'rms': 3.32, 'linf': 40.6, 'fora': 0.994}}
alt.update(alteracao)

print('\n' + '=' * 78)
print('SONDA DE AMPLITUDE - média das 3 últimas épocas; alteração medida no ckpt final')
print('=' * 78)
print(f"{'':<20}" + ''.join(f'{k:>14}' for k in res))
for rotulo, chave, fmt in (('AUC efetiva', 'auc', '14.4f'), ('acurácia', 'acc', '14.4f'),
                           ('PSNR (dB)', 'psnr', '14.2f')):
    print(f'{rotulo:<20}' + ''.join(f'{media3(c[chave]):{fmt}}' if c else f"{'--':>14}"
                                    for c in res.values()))
for rotulo, chave, fmt in (('taxa de alteração', 'taxa', '14.1%'), ('RMS (níveis)', 'rms', '14.2f'),
                           ('L∞ (níveis)', 'linf', '14.1f'), ('fora da grade', 'fora', '14.1%')):
    print(f'{rotulo:<20}' + ''.join(f'{a[chave]:{fmt}}' if a else f"{'--':>14}"
                                    for a in (alt.get(k) for k in res)))

for nome in res:
    if nome == 'uniforme' or not res[nome]:
        continue
    c = res[nome]
    print(f'\n  {nome}, por época')
    print('  AUCef ' + ' '.join(f'{a:6.4f}' for a in c['auc']))
    print('  PSNR  ' + ' '.join(f'{p:6.2f}' for p in c['psnr']))
    print('  acc   ' + ' '.join(f'{a:6.4f}' for a in c['acc']))

print('\n' + '=' * 78)
def veredito(nome, texto_move, texto_nao):
    c = res.get(nome)
    if not c:
        return
    auc, acc = media3(c['auc']), media3(c['acc'])
    if acc < 0.99:
        print(f'[{nome}] AUC {auc:.4f}, mas LEITURA QUEBRADA ({acc:.4f} < 0,99): a AUC não vale')
        print('   nada enquanto a mensagem não é recuperada - o gerador pode estar enganando o')
        print('   detector por deixar de esconder. Mais épocas ou lambda_mensagem maior neste braço.')
        return
    print(f'[{nome}] AUC {auc:.4f}, leitura ok ({acc:.4f}):')
    print('   ' + (texto_move if auc < 0.9990 else texto_nao))

veredito('quant',
         'A GRADE PESAVA. Parte da AUC 1,0 de todos os runs anteriores era a estego fora de k/255. '
         'Daqui em diante, todo treino quantiza.',
         'a grade não explica a AUC 1,0. Quantizar continua obrigatório (é o que uma estego salva em disco sofre).')
veredito('quant_t2',
         'A AMPLITUDE PESA: com ±2 níveis e na grade, a SRNet cede.',
         'nem ±2 níveis basta com ~79% dos pixels alterados. Olhe a taxa de alteração.')
veredito('quant_t2_l1',
         'A ESPARSIDADE PESA: poucos pixels alterados é o que move o detector. '
         'Próximo: custo aprendido de ONDE alterar (a Fase 6 volta, agora sobre a esparsidade).',
         'nem esparsificando. Conferir a taxa de alteração: se ainda estiver alta, subir '
         'lambda_residuo; se já estiver baixa, o problema é o padrão espacial, não a quantidade.')
print('A medida definitiva continua sendo a Fase 4 (P_E com detector independente).')
print('=' * 78)

## 6.11 Esparsidade estrutural
Só os 8 pixels de maior textura de cada bloco 8×8 podem mudar (braços `esp8` e `esp8_t2`).

In [ ]:
import importlib, os, csv, glob
import torch
import fase0_codificador_decodificador, fase2_wgan_gp, fase25_residual
import fase3_esteganalisador as f3
import fase6_adaptativo as f6
import fase6_amplitude as fa

EPOCAS = 10
LAMBDA = 0.05
MSG    = 16
BRACOS = [
    ('esp8',    None, 8),
    ('esp8_t2', 2,    8),
]
pasta_boss = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01')


def medir(saida):
    ck = next((os.path.join(saida, n) for n in ('ckpt_last.pt', 'ckpt_best.pt')
               if os.path.exists(os.path.join(saida, n))), None)
    if ck is None:
        return None
    estado = torch.load(ck, map_location='cpu', weights_only=False)
    cfg = dict(estado['cfg'])
    cod = fa.CodificadorAmplitude(cfg['canais'], cfg['profundidade_dados'], cfg['largura'],
                                  fase25_residual.BLOCOS_RESIDUAIS).to(cfg['dispositivo'])
    cod.load_state_dict(estado['codificador']); cod.eval()
    caminhos = sorted(p for p in glob.glob(os.path.join(pasta_boss, '**', '*'), recursive=True)
                      if p.lower().endswith(fase0_codificador_decodificador.EXTENSOES_IMG))[-256:]
    ds = fase0_codificador_decodificador.DatasetImagens(caminhos, 1, cfg['tamanho_img'], treino=False)
    x = torch.stack([ds[i] for i in range(len(ds))]).to(cfg['dispositivo'])
    torch.manual_seed(0)
    with torch.no_grad():
        msg = fase0_codificador_decodificador.mensagem_aleatoria(len(x), cfg)
        y = cod(x, fase0_codificador_decodificador.espalhar_mensagem(msg, cfg))
    return fa.medir_alteracao(x, y)


alteracao = {}
for nome, teto, k in BRACOS:
    saida   = os.path.join(PASTA_TRABALHO, f'phase6_esp_{nome}')
    espelho = os.path.join(PASTA_SAIDA_BASE, f'phase6_esp_{nome}')
    for m in (fase0_codificador_decodificador, fase2_wgan_gp, fase25_residual, f3, f6, fa):
        importlib.reload(m)
    fa.usar('amplitude', niveis=teto, quantizar=True, k=k, bloco=128 // MSG)

    feito = os.path.join(espelho, 'metricas_val.csv')
    if os.path.exists(feito) and sum(1 for _ in open(feito, encoding='utf-8')) > EPOCAS:
        print(f'[pulado] {nome}: {EPOCAS} épocas já registradas em {espelho}')
    else:
        f3.CONFIG.update(f3.COLAB_GPU)
        f3.CONFIG.update(
            pasta_dados     = pasta_boss,
            canais          = 1,
            carregar_fase2  = CKPT_FASE2,
            pasta_saida     = saida,
            espelho_ckpt    = espelho,
            max_imagens     = None,
            epocas          = EPOCAS,
            srnet_compacta  = False,
            alvo_srnet      = 'equilibrio',
            lambda_srnet    = LAMBDA,
            tamanho_msg     = MSG,
            arquitetura     = 'amplitude',
            retomar_de      = None,
        )
        print('=' * 78)
        print(f'SONDA DE ESPARSIDADE - braço {nome}: {k} de 64 pixels por bloco, teto={teto}  ->  {saida}')
        print('=' * 78)
        f3.treinar()
        fase25_residual.espelhar(saida, espelho)

    alteracao[nome] = medir(saida) or medir(espelho)
    fa.usar('residual')


def curva(*pastas):
    for pasta in pastas:
        caminho = os.path.join(pasta, 'metricas_val.csv')
        if os.path.exists(caminho):
            with open(caminho, encoding='utf-8') as f:
                linhas = list(csv.DictReader(f))
            return {'auc':  [max(float(l['auc_srnet']), 1 - float(l['auc_srnet'])) for l in linhas],
                    'psnr': [float(l['psnr']) for l in linhas],
                    'acc':  [float(l['acuracia_bits']) for l in linhas]}
    return None

def ambos(nome):
    return (os.path.join(PASTA_TRABALHO, nome), os.path.join(PASTA_SAIDA_BASE, nome))

media3 = lambda v: sum(v[-3:]) / len(v[-3:])
res = {'controle256': curva(*ambos(f'phase3_carga_msg{MSG}'))}
res.update({nome: curva(*ambos(f'phase6_esp_{nome}')) for nome, _, _ in BRACOS})
alt = {'controle256': {'taxa': 0.775, 'rms': 3.24, 'linf': 44.0, 'fora': 0.994}}
alt.update(alteracao)

print('\n' + '=' * 78)
print('SONDA DE ESPARSIDADE - 256 bits; média das 3 últimas épocas; alteração no ckpt final')
print('=' * 78)
print(f"{'':<20}" + ''.join(f'{k:>14}' for k in res))
for rotulo, chave, fmt in (('AUC efetiva', 'auc', '14.4f'), ('acurácia', 'acc', '14.4f'),
                           ('PSNR (dB)', 'psnr', '14.2f')):
    print(f'{rotulo:<20}' + ''.join(f'{media3(c[chave]):{fmt}}' if c else f"{'--':>14}"
                                    for c in res.values()))
for rotulo, chave, fmt in (('taxa de alteração', 'taxa', '14.1%'), ('RMS (níveis)', 'rms', '14.2f'),
                           ('L∞ (níveis)', 'linf', '14.1f'), ('fora da grade', 'fora', '14.1%')):
    print(f'{rotulo:<20}' + ''.join(f'{a[chave]:{fmt}}' if a else f"{'--':>14}"
                                    for a in (alt.get(k) for k in res)))

for nome in res:
    if nome == 'controle256' or not res[nome]:
        continue
    c = res[nome]
    print(f'\n  {nome}, por época')
    print('  AUCef ' + ' '.join(f'{a:6.4f}' for a in c['auc']))
    print('  PSNR  ' + ' '.join(f'{p:6.2f}' for p in c['psnr']))
    print('  acc   ' + ' '.join(f'{a:6.4f}' for a in c['acc']))

print('\n' + '=' * 78)
for nome, teto, k in BRACOS:
    c, a = res.get(nome), alteracao.get(nome)
    if not c:
        continue
    auc, acc = media3(c['auc']), media3(c['acc'])
    if acc < 0.99:
        print(f'[{nome}] AUC {auc:.4f}, mas LEITURA QUEBRADA ({acc:.4f} < 0,99): a AUC não vale.')
        print(f'   Com {k} pixels por bit a rede não recuperou a leitura em {EPOCAS} épocas.')
        print('   Próximo: mais épocas, ou k maior (16 de 64) antes de concluir.')
    elif auc < 0.9990:
        print(f'[{nome}] AUC {auc:.4f} com leitura ok ({acc:.4f}): A ESPARSIDADE MOVE O DETECTOR.')
        print('   É o primeiro braço do projeto que cede. Próximo: varrer k (4, 8, 16) e levar o')
        print('   melhor para a Fase 4 - a AUC do laço é triagem, P_E é a medida.')
    else:
        extra = (f' L∞ final {a["linf"]:.0f} níveis: a rede compensou com amplitude.'
                 if a and teto is None and a['linf'] > 10 else '')
        print(f'[{nome}] AUC {auc:.4f} com leitura ok ({acc:.4f}): nem com {k / 64:.1%} dos pixels.{extra}')
print('A medida definitiva continua sendo a Fase 4 (P_E com detector independente).')
print('=' * 78)

## 7. Treino completo
60 épocas no dataset inteiro, retomável pelo `ckpt_last.pt`.

In [ ]:
import importlib, os
import fase25_residual as f25

SAIDA_P3   = os.path.join(PASTA_TRABALHO, 'phase3_bossbase')
ESPELHO_P3 = os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase')

LAMBDA_FINAL = 0.05
EPOCAS_P3    = 30

RETOMAR = os.path.join(SAIDA_P3, 'ckpt_last.pt')
import fase0_codificador_decodificador, fase2_wgan_gp, fase3_esteganalisador as f3
for m in (fase0_codificador_decodificador, fase2_wgan_gp, f3):
    importlib.reload(m)

f3.CONFIG.update(f3.COLAB_GPU)
f3.CONFIG.update(
    pasta_dados     = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
    canais          = 1,
    carregar_fase2  = CKPT_FASE2,
    pasta_saida     = SAIDA_P3,
    espelho_ckpt    = ESPELHO_P3,
    max_imagens     = None,
    epocas          = EPOCAS_P3,
    retomar_de      = RETOMAR if os.path.exists(RETOMAR) else None,
    srnet_compacta  = False,
    lambda_srnet    = LAMBDA_FINAL,
    alvo_srnet      = 'equilibrio',
)
resultado_p3 = f3.treinar()
print('\nRESULTADO FASE 3:', resultado_p3)

f25.espelhar(SAIDA_P3, ESPELHO_P3)


## 8. Amostras
Mostra portadora, estego e resíduo.

In [ ]:
import os
PASTA_P3 = next((p for p in (os.path.join(globals().get('PASTA_TRABALHO', '/content/runs'), 'phase3_bossbase'),
                                       os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase'))
                  if os.path.isdir(p)), os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase'))
import glob, os
from IPython.display import Image as IPyImage, display
amostras = sorted(glob.glob(os.path.join(PASTA_P3, 'samples_epoch*.png')))
if amostras:
    print('Última amostra:', amostras[-1], ' (linha 1=portadora, 2=estego, 3=resíduo)')
    display(IPyImage(filename=amostras[-1]))
else:
    print('Nenhuma amostra ainda - rode o treino primeiro.')


## 9. Curvas
Plota qualidade, detecção e perdas a partir dos CSVs.

In [ ]:
import os
PASTA_P3 = next((p for p in (os.path.join(globals().get('PASTA_TRABALHO', '/content/runs'), 'phase3_bossbase'),
                                       os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase'))
                  if os.path.isdir(p)), os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase'))
import csv, os
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

pasta = PASTA_P3

def coluna(arquivo, *nomes):
    saida = {n: [] for n in nomes}
    with open(os.path.join(pasta, arquivo)) as f:
        for r in csv.DictReader(f):
            for n in nomes:
                saida[n].append(float(r[n]))
    return saida

v = coluna('metricas_val.csv', 'epoca', 'psnr', 'auc_srnet', 'acuracia_bits')
t = coluna('metricas_treino.csv', 'epoca', 'srnet_bce', 'enganar')

fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(7.5, 7.5), sharex=True, dpi=140,
                                 gridspec_kw={'height_ratios': [1.3, 1, 1]})
a1.plot(v['epoca'], v['psnr'], color='#2a78d6', label='PSNR')
a1.axhline(ALVO_PSNR, ls='--', lw=1, color='#eb6834', label=f'piso (partida - 1 dB = {ALVO_PSNR:.2f})')
a1.set_ylabel('PSNR (dB)'); a1.grid(alpha=.3); a1.legend(fontsize=8)
a1.set_title('Fase 3 - qualidade, detecção e saúde do treino')

a2.plot(v['epoca'], v['auc_srnet'], color='#0ca30c')
a2.axhline(0.5, ls='--', lw=1, color='#898781')
a2.axhspan(0.45, 0.55, color='#0ca30c', alpha=.08)
a2.set_ylabel('AUC (laço)'); a2.set_ylim(0, 1); a2.grid(alpha=.3)

a3.plot(t['epoca'], t['srnet_bce'], color='#8b5cf6', label='srnet_bce (detector aprendendo?)')
a3.plot(t['epoca'], t['enganar'], color='#eb6834', label='enganar (termo dominando?)')
a3.axhline(0.693, ls=':', lw=1, color='#898781')
a3.text(v['epoca'][0], 0.72, 'ln 2 = piso de ambas', fontsize=7, color='#898781')
a3.set_ylabel('perda'); a3.set_xlabel('época'); a3.grid(alpha=.3); a3.legend(fontsize=8)
fig.tight_layout()
saida = os.path.join(pasta, 'curva_fase3.png')
fig.savefig(saida, facecolor='white')
print('Salvo em', saida)
display(IPyImage(filename=saida))


## 10. Conferência no Drive
Verifica checkpoints, log, CSVs e curva no Drive.

In [ ]:
import os
PASTA_P3 = next((p for p in (os.path.join(globals().get('PASTA_TRABALHO', '/content/runs'), 'phase3_bossbase'),
                                       os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase'))
                  if os.path.isdir(p)), os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase'))
import os, glob
pasta = PASTA_P3
for f in sorted(glob.glob(os.path.join(pasta, '*'))):
    print('%10.1f KB  %s' % (os.path.getsize(f) / 1024, os.path.basename(f)))


## 11. Fase 4 - P_E com detectores independentes
Treina `srnet`, `srnet_cur` e `effnet` do zero sobre os geradores congelados e o LSBM, e mede P_E.

In [ ]:
import importlib, os, subprocess, sys
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)
import fase0_codificador_decodificador, fase25_residual, fase3_esteganalisador
import fase6_adaptativo, fase6_amplitude, fase4_pe as f4
for m in (fase0_codificador_decodificador, fase25_residual, fase3_esteganalisador,
          fase6_adaptativo, fase6_amplitude, f4):
    importlib.reload(m)

def ckpt(nome, arquivo='ckpt_last.pt'):
    return os.path.join(PASTA_SAIDA_BASE, nome, arquivo)

GERADORES = [
    {'nome': 'fase25',       'tipo': 'residual',  'ckpt': CKPT_FASE2},
    {'nome': 'fase3_lam005', 'tipo': 'residual',  'ckpt': ckpt('phase3_sonda_lam0.05')},
    {'nome': 'quant',        'tipo': 'amplitude', 'ckpt': ckpt('phase6_amp_quant')},
    {'nome': 'lsbm',         'tipo': 'lsbm',      'bpp': 0.0625},
]
DETECTORES = ('srnet', 'effnet', 'srnet_cur')

faltando = [g['nome'] for g in GERADORES if g['tipo'] != 'lsbm' and not os.path.exists(g['ckpt'])]
for nome in faltando:
    print(f'[aviso] checkpoint de {nome!r} não encontrado no Drive - gerador pulado')
GERADORES = [g for g in GERADORES if g['nome'] not in faltando]

linhas_f4 = f4.rodar(GERADORES, DETECTORES, dict(
    pasta_dados = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
    pasta_saida = os.path.join(PASTA_TRABALHO, 'phase4'),
    espelho     = os.path.join(PASTA_SAIDA_BASE, 'phase4'),
    epocas      = 20,
))

### 11.1 HILL
Mede o P_E do HILL a 0,0625 e 0,4 bpp com `effnet` e `srnet_cur`.

In [ ]:
import importlib
importlib.reload(f4)

GERADORES_HILL = [
    {'nome': 'hill',   'tipo': 'hill', 'bpp': 0.0625},
    {'nome': 'hill04', 'tipo': 'hill', 'bpp': 0.4},
]
linhas_f4 = f4.rodar(GERADORES + GERADORES_HILL, ('effnet', 'srnet_cur'), dict(
    pasta_dados = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
    pasta_saida = os.path.join(PASTA_TRABALHO, 'phase4'),
    espelho     = os.path.join(PASTA_SAIDA_BASE, 'phase4'),
    epocas      = 20,
))